In [25]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

# -------------------------
# Paths (raw strings to avoid \D warnings)
# -------------------------
VDI_DIR  = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
LULC_DIR = Path(r"C:\Drought\Land Cover")

f_ESI  = VDI_DIR / "ESI_dryz.nc"   # var: ESI_dryz
f_SIF  = VDI_DIR / "SIF_dryz.nc"   # var: SIF_dryz
f_SM   = VDI_DIR / "SMroot_dryz.nc"  # var: SMroot_dryz
f_VOD  = VDI_DIR / "VOD_dryz.nc"   # var: VOD_dryz
f_TVDI = VDI_DIR / "TVDI_Z_monthly_India_0p05deg_2001-2023.nc"  # var: TVDI_Z

f_forest = LULC_DIR / "lulc_forest_frac_0p05deg.nc"
f_shrub  = LULC_DIR / "lulc_shrub_grass_frac_0p05deg.nc"
f_crop   = LULC_DIR / "lulc_cropland_frac_0p05deg.nc"

# -------------------------
# Helpers
# -------------------------
def open_var(path: Path, preferred_var: str = None) -> xr.DataArray:
    ds = xr.open_dataset(path)
    # pick variable
    if preferred_var and preferred_var in ds:
        da = ds[preferred_var]
    else:
        # fall back to first data var
        v = list(ds.data_vars)[0]
        da = ds[v]
    # unify dimension names to (time, lat, lon) where present
    ren = {}
    for d in da.dims:
        dlow = d.lower()
        if dlow in ("y", "latitude", "lat"):
            ren[d] = "lat"
        elif dlow in ("x", "longitude", "lon"):
            ren[d] = "lon"
        elif dlow == "month":  # keep 'time' as-is; month may exist in some files alongside time
            # if this is an extra coord (e.g., TVDI_mu), we'll ignore that var anyway
            pass
    if ren:
        da = da.rename(ren)
    # put time first if exists
    order = [d for d in ("time", "lat", "lon") if d in da.dims]
    da = da.transpose(*order, ...)
    return da

def to_month_end(da: xr.DataArray) -> xr.DataArray:
    if "time" not in da.dims:
        return da
    t = pd.to_datetime(da.time.values)
    t_end = pd.to_datetime([pd.Timestamp(tt).to_period("M").asfreq("M").to_timestamp() for tt in t])
    return da.assign_coords(time=t_end)

def align_to_ref_grid(da: xr.DataArray, ref_lat: xr.DataArray, ref_lon: xr.DataArray) -> xr.DataArray:
    """
    Align da to reference (lat, lon) using nearest-neighbor interpolation.
    This avoids shape mismatch (e.g., 605 vs 611) and slight coordinate drift.
    """
    # ensure we have lat/lon
    if "lat" not in da.dims or "lon" not in da.dims:
        raise ValueError(f"DataArray missing lat/lon dims: {da.dims}")
    # interp expects increasing coords
    da_sorted = da.sortby(["lat", "lon"])
    ref_lat_sorted = ref_lat.sortby(ref_lat)
    ref_lon_sorted = ref_lon.sortby(ref_lon)
    # nearest neighbor to keep Z-scale; set fill_value to NaN outside overlap
    out = da_sorted.interp(
        lat=ref_lat_sorted, lon=ref_lon_sorted,
        method="nearest", kwargs={"fill_value": np.nan}
    )
    return out

def open_fraction(path: Path, ref_lat: xr.DataArray, ref_lon: xr.DataArray) -> xr.DataArray:
    ds = xr.open_dataset(path)
    var = "fraction" if "fraction" in ds else list(ds.data_vars)[0]
    da = ds[var]
    # standardize dims
    ren = {}
    for d in da.dims:
        dlow = d.lower()
        if dlow in ("y", "latitude", "lat"):
            ren[d] = "lat"
        elif dlow in ("x", "longitude", "lon"):
            ren[d] = "lon"
    if ren:
        da = da.rename(ren)
    da = da.transpose("lat", "lon")
    # align to ref grid
    da = align_to_ref_grid(da, ref_lat, ref_lon)
    return da.fillna(0.0).astype("float32")

def stress_clip(da: xr.DataArray, sgn: int) -> xr.DataArray:
    return (sgn * da).clip(min=-3.0, max=3.0)

# -------------------------
# Load fields
# -------------------------
ESI  = open_var(f_ESI,  "ESI_dryz")
SIF  = open_var(f_SIF,  "SIF_dryz")
SMRZ = open_var(f_SM,   "SMroot_dryz")
VOD  = open_var(f_VOD,  "VOD_dryz")
TVDI = open_var(f_TVDI, "TVDI_Z")   # only the TVDI_Z var, ignore mu/sig

# month-end + common time intersection
ESI  = to_month_end(ESI)
SIF  = to_month_end(SIF)
SMRZ = to_month_end(SMRZ)
VOD  = to_month_end(VOD)
TVDI = to_month_end(TVDI)

common_time = sorted(set(ESI.time.values) &
                     set(SIF.time.values) &
                     set(SMRZ.time.values) &
                     set(VOD.time.values) &
                     set(TVDI.time.values))
ESI  = ESI.sel(time=common_time)
SIF  = SIF.sel(time=common_time)
SMRZ = SMRZ.sel(time=common_time)
VOD  = VOD.sel(time=common_time)
TVDI = TVDI.sel(time=common_time)

# pick a reference grid (ESI's lat/lon)
ref_lat = ESI["lat"]
ref_lon = ESI["lon"]

# Align all to the same grid (fixes 605 vs 611 mismatch)
SIF  = align_to_ref_grid(SIF,  ref_lat, ref_lon)
SMRZ = align_to_ref_grid(SMRZ, ref_lat, ref_lon)
VOD  = align_to_ref_grid(VOD,  ref_lat, ref_lon)
TVDI = align_to_ref_grid(TVDI, ref_lat, ref_lon)

# -------------------------
# Verify/force stress sign (higher = drier), clip to [-3,3]
# Your *_dryz and TVDI_Z are already stress → keep +1.
# If any needs flipping later, change +1 to -1.
# -------------------------
ESI_s  = stress_clip(ESI,  +1)
SMRZ_s = stress_clip(SMRZ, +1)
SIF_s  = stress_clip(SIF,  +1)
VOD_s  = stress_clip(VOD,  +1)
TVDI_s = stress_clip(TVDI, +1)

# -------------------------
# Biome fractions → dominant biome mask
# -------------------------
frac_forest = open_fraction(f_forest, ref_lat, ref_lon)
frac_shrub  = open_fraction(f_shrub,  ref_lat, ref_lon)
frac_crop   = open_fraction(f_crop,   ref_lat, ref_lon)

stacked = xr.concat([frac_forest, frac_shrub, frac_crop], dim="biome")  # 0=forest,1=shrub,2=crop
dom_idx = stacked.argmax("biome")
none_mask = (stacked.sum("biome") <= 0)
dom_idx = dom_idx.where(~none_mask)

biome_code = xr.full_like(frac_forest, np.nan, dtype="float32")
biome_code = xr.where(dom_idx==0, 1.0, biome_code)  # forest
biome_code = xr.where(dom_idx==1, 2.0, biome_code)  # shrub/grass
biome_code = xr.where(dom_idx==2, 3.0, biome_code)  # cropland
biome_code.name = "biome_code"

mask_forest = xr.where(biome_code==1.0, 1.0, 0.0)
mask_shrub  = xr.where(biome_code==2.0, 1.0, 0.0)
mask_crop   = xr.where(biome_code==3.0, 1.0, 0.0)

# -------------------------
# Weights per biome (order: [ESI, SMRZ, SIF, TVDI, VOD])
# -------------------------
wts_cropland = np.array([0.30, 0.25, 0.20, 0.15, 0.10], dtype="float32")
wts_forest   = np.array([0.15, 0.20, 0.25, 0.05, 0.35], dtype="float32")
wts_shrub    = np.array([0.30, 0.20, 0.15, 0.25, 0.10], dtype="float32")

comp_names = ["ESI","SMRZ","SIF","TVDI","VOD"]
W = xr.zeros_like(frac_forest).expand_dims(comp=comp_names).astype("float32")

# start with cropland everywhere
for name, val in zip(comp_names, wts_cropland):
    W.loc[dict(comp=name)] = val
# overwrite where forest/shrub masks are 1
for wrow, mask in [(wts_forest, mask_forest), (wts_shrub, mask_shrub)]:
    for name, val in zip(comp_names, wrow):
        W.loc[dict(comp=name)] = xr.where(mask==1.0, val, W.sel(comp=name))

# -------------------------
# Stack components Z[time, comp, lat, lon]
# -------------------------
Z = xr.concat([ESI_s, SMRZ_s, SIF_s, TVDI_s, VOD_s],
              dim=xr.DataArray(comp_names, dims="comp")).transpose("time","comp","lat","lon")

valid = xr.where(np.isfinite(Z), 1.0, 0.0)
num = (W.expand_dims(time=Z.time) * xr.where(np.isfinite(Z), Z, 0.0)).sum(dim="comp")
den = (W.expand_dims(time=Z.time) * valid).sum(dim="comp")
VDI_all = xr.where(den > 0, num / den, np.nan).astype("float32")
VDI_all.name = "VDI"

VDI_forest = VDI_all.where(mask_forest==1.0)
VDI_shrub  = VDI_all.where(mask_shrub==1.0)
VDI_crop   = VDI_all.where(mask_crop==1.0)

# -------------------------
# Save
# -------------------------
def save_nc(da: xr.DataArray, path: Path, title: str):
    ds = xr.Dataset({"VDI": da})
    ds["biome_code"] = biome_code
    ds["VDI"].attrs.update({
        "long_name": "Vegetation Drought Index (biome-weighted composite)",
        "units": "1 (higher=drier)",
        "components": "ESI_dryz, SMroot_dryz, SIF_dryz, TVDI_Z, VOD_dryz",
        "z_clip": "components clipped to [-3,3] before fusion",
        "weights": "Cropland[ESI,SMRZ,SIF,TVDI,VOD]=[0.30,0.25,0.20,0.15,0.10]; "
                   "Forest=[0.15,0.20,0.25,0.05,0.35]; Shrub/Grass=[0.30,0.20,0.15,0.25,0.10]; "
                   "renormalised over available components",
    })
    ds.attrs.update({
        "title": title,
        "Conventions": "CF-1.8",
        "history": "Monthly fusion at 0.05°, aligned to ESI grid; nearest-neighbor interp for alignment.",
    })
    enc = {
        "VDI": {"zlib": True, "complevel": 4, "dtype": "float32", "chunksizes": (12, 128, 128)},
        "biome_code": {"zlib": True, "complevel": 4, "dtype": "float32", "chunksizes": (128, 128)},
        "lat": {"dtype": "float32"},
        "lon": {"dtype": "float32"},
    }
    ds.to_netcdf(path, encoding=enc)

save_nc(VDI_all,    VDI_DIR / "VDI_allbiomes.nc",   "VDI (biome-weighted, India, 0.05°)")
save_nc(VDI_forest, VDI_DIR / "VDI_forest.nc",      "VDI (forest mask, India, 0.05°)")
save_nc(VDI_shrub,  VDI_DIR / "VDI_shrub_grass.nc", "VDI (shrub+grass mask, India, 0.05°)")
save_nc(VDI_crop,   VDI_DIR / "VDI_cropland.nc",    "VDI (cropland mask, India, 0.05°)")

print("Done. Wrote:")
print(" -", VDI_DIR / "VDI_allbiomes.nc")
print(" -", VDI_DIR / "VDI_forest.nc")
print(" -", VDI_DIR / "VDI_shrub_grass.nc")
print(" -", VDI_DIR / "VDI_cropland.nc")


Done. Wrote:
 - C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_allbiomes.nc
 - C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_forest.nc
 - C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_shrub_grass.nc
 - C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_cropland.nc


In [27]:
from pathlib import Path
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap, ScalarMappable

# -------------------------
# Paths
# -------------------------
VDI_DIR = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
INDIA_SHP = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

FILES = [
    (VDI_DIR / "VDI_allbiomes.nc",  "VDI (All Biomes)",           "VDI_allbiomes_clim.png"),
    (VDI_DIR / "VDI_forest.nc",     "VDI (Forest)",               "VDI_forest_clim.png"),
    (VDI_DIR / "VDI_shrub_grass.nc","VDI (Shrub + Grasslands)",   "VDI_shrub_grass_clim.png"),
    (VDI_DIR / "VDI_cropland.nc",   "VDI (Cropland)",             "VDI_cropland_clim.png"),
]

# -------------------------
# Load India boundary (EPSG:4326)
# -------------------------
gdf = gpd.read_file(INDIA_SHP)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS; please define it (should be EPSG:4326).")
gdf = gdf.to_crs("EPSG:4326")

# -------------------------
# Helper: monthly climatology plot
# -------------------------
def plot_monthly_climatology(nc_path: Path, fig_title: str, out_png: Path):
    ds = xr.open_dataset(nc_path)
    if "VDI" not in ds:
        # first data var if not named VDI
        vname = list(ds.data_vars)[0]
        da = ds[vname]
    else:
        da = ds["VDI"]

    # Expect dims (time, lat, lon). Sort to be safe.
    if "time" not in da.dims:
        raise ValueError(f"{nc_path.name}: variable has no 'time' dimension.")
    da = da.sortby(["lat", "lon"])

    # Monthly climatology (mean over years)
    clim = da.groupby("time.month").mean("time", skipna=True)

    # Grid extent for imshow
    lat = clim["lat"].values
    lon = clim["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]

    # Common color scale across months (robust percentiles)
    vmin, vmax = np.nanpercentile(clim.values, [2, 98])
    # Ensure some spread; fall back if they collapse
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = -2.5, 2.5

    # Colormap (rainbow) with NaNs shown as white
    cmap = get_cmap("rainbow").copy()
    cmap.set_bad("white")
    norm = Normalize(vmin=vmin, vmax=vmax)

    # Figure: 3x4 grid (12 months)
    fig, axes = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True, facecolor="white")
    axes = axes.ravel()

    # Month names
    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    for m in range(12):
        ax = axes[m]
        frame = clim.sel(month=m+1)
        img = ax.imshow(
            frame.values,
            origin="lower",  # lat is ascending; lower = south
            extent=extent,
            cmap=cmap,
            norm=norm,
            interpolation="nearest",
        )
        # Overlay India boundary
        gdf.boundary.plot(ax=ax, color="k", linewidth=0.6)

        # Titles per panel
        ax.set_title(month_names[m], fontsize=11, color="black")
        # Hide axes (no lat/lon)
        ax.set_axis_off()
        # Clean white background
        ax.set_facecolor("white")

    # Turn off any unused axes if present (shouldn’t be, but be safe)
    for k in range(12, len(axes)):
        axes[k].set_visible(False)

    # Single colorbar on the RIGHT
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), orientation="vertical", fraction=0.02, pad=0.02)
    # "just write of colour bar" – keep ticks, no label text
    cbar.set_label("")

    # Figure title (optional, subtle)
    fig.suptitle(fig_title, fontsize=14, color="black")

    # Save
    out_path = VDI_DIR / out_png
    fig.savefig(out_path, dpi=300, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)

# -------------------------
# Make all four figures
# -------------------------
for nc_path, title, png_name in FILES:
    plot_monthly_climatology(nc_path, title, png_name)


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2261637279.py:75: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_allbiomes_clim.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2261637279.py:75: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_forest_clim.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2261637279.py:75: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_shrub_grass_clim.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2261637279.py:75: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_cropland_clim.png


In [29]:
from pathlib import Path
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap, ScalarMappable

# -------------------------
# Paths
# -------------------------
VDI_DIR = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
INDIA_SHP = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

FILES = [
    (VDI_DIR / "VDI_allbiomes.nc",  "VDI (All Biomes)",           "VDI_allbiomes_{year}_monthly.png"),
    (VDI_DIR / "VDI_forest.nc",     "VDI (Forest)",               "VDI_forest_{year}_monthly.png"),
    (VDI_DIR / "VDI_shrub_grass.nc","VDI (Shrub + Grasslands)",   "VDI_shrub_grass_{year}_monthly.png"),
    (VDI_DIR / "VDI_cropland.nc",   "VDI (Cropland)",             "VDI_cropland_{year}_monthly.png"),
]

YEARS = [2002, 2005, 2009, 2015, 2019]

# Choose color scale strategy:
#   (a) fixed for comparability across all figures:
VMIN, VMAX = -2.5, 2.5
#   (b) OR uncomment to compute per-figure robust scale:
# VMIN = VMAX = None

# -------------------------
# Load India boundary (EPSG:4326)
# -------------------------
gdf = gpd.read_file(INDIA_SHP)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS; please define it (should be EPSG:4326).")
gdf = gdf.to_crs("EPSG:4326")

# -------------------------
# Helper: draw 12 monthly panels for a given year
# -------------------------
def plot_year_monthly(nc_path: Path, fig_title: str, out_tmpl: str, year: int):
    ds = xr.open_dataset(nc_path)
    vname = "VDI" if "VDI" in ds else list(ds.data_vars)[0]
    da = ds[vname].sortby(["lat", "lon"])

    # Subset this year
    if "time" not in da.dims:
        raise ValueError(f"{nc_path.name}: variable has no 'time' dimension.")
    sub = da.sel(time=da["time"].dt.year == year)
    if sub.sizes.get("time", 0) == 0:
        print(f"Warning: {nc_path.name} has no data for {year}. Skipping.")
        return

    # Build a (month, lat, lon) cube ensuring Jan..Dec order (fill missing months with NaN)
    months = np.arange(1, 13, dtype=int)
    frames = []
    for m in months:
        msel = sub.sel(time=sub["time"].dt.month == m)
        if msel.sizes["time"] >= 1:
            frames.append(msel.mean("time", skipna=True))
        else:
            # empty month -> NaNs on the same grid
            frames.append(da.isel(time=0, drop=True)*np.nan)
    cube = xr.concat(frames, dim=("month"))
    cube = cube.assign_coords(month=("month", months))

    # Extent for imshow
    lat = cube["lat"].values
    lon = cube["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]

    # Color scale
    if VMIN is None or VMAX is None:
        vmin, vmax = np.nanpercentile(cube.values, [2, 98])
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = -2.5, 2.5
    else:
        vmin, vmax = VMIN, VMAX

    cmap = get_cmap("rainbow").copy()
    cmap.set_bad("white")
    norm = Normalize(vmin=vmin, vmax=vmax)

    # Figure (3x4 panels)
    fig, axes = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True, facecolor="white")
    axes = axes.ravel()
    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    for i, m in enumerate(months):
        ax = axes[i]
        frame = cube.sel(month=m)
        img = ax.imshow(
            frame.values,
            origin="lower",
            extent=extent,
            cmap=cmap,
            norm=norm,
            interpolation="nearest",
        )
        gdf.boundary.plot(ax=ax, color="k", linewidth=0.6)
        ax.set_title(month_names[i], fontsize=11, color="black")
        ax.set_axis_off()
        ax.set_facecolor("white")

    # Single colorbar on the right
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), orientation="vertical", fraction=0.02, pad=0.02)
    cbar.set_label("")  # no label text; ticks only

    # Title
    fig.suptitle(f"{fig_title} — {year}", fontsize=14, color="black")

    out_png = VDI_DIR / out_tmpl.format(year=year)
    fig.savefig(out_png, dpi=300, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_png)

# -------------------------
# Generate all figures
# -------------------------
for year in YEARS:
    for nc_path, title, tmpl in FILES:
        plot_year_monthly(nc_path, title, tmpl, year)


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1137564995.py:95: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_allbiomes_2002_monthly.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1137564995.py:95: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_forest_2002_monthly.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1137564995.py:95: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


...

Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_forest_2019_monthly.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1137564995.py:95: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_shrub_grass_2019_monthly.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1137564995.py:95: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_cropland_2019_monthly.png


In [33]:
from pathlib import Path
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable

# -------------------------
# Paths
# -------------------------
VDI_DIR = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
INDIA_SHP = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

FILES = [
    (VDI_DIR / "VDI_allbiomes.nc",   "VDI (All Biomes)",           "VDI_allbiomes_{year}_classes.png"),
    (VDI_DIR / "VDI_forest.nc",      "VDI (Forest)",               "VDI_forest_{year}_classes.png"),
    (VDI_DIR / "VDI_shrub_grass.nc", "VDI (Shrub + Grasslands)",   "VDI_shrub_grass_{year}_classes.png"),
    (VDI_DIR / "VDI_cropland.nc",    "VDI (Cropland)",             "VDI_cropland_{year}_classes.png"),
]

YEARS = [2002, 2005, 2009, 2015, 2019]

# -------------------------
# India outline
# -------------------------
gdf = gpd.read_file(INDIA_SHP)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS; please define it (should be EPSG:4326).")
gdf = gdf.to_crs("EPSG:4326")

# -------------------------
# Class thresholds (Z-based) → integer codes 0..5
# -------------------------
# Order is important: check from most severe upward.
def classify_to_int(da: xr.DataArray) -> xr.DataArray:
    arr = da.values
    cls = np.full(arr.shape, np.nan, dtype="float32")
    mask = ~np.isnan(arr)
    a = arr[mask]

    # Build classes
    out = np.full(a.shape, 5, dtype="int16")  # default near-normal
    out[a <= -0.5] = 4
    out[a <= -1.0] = 3
    out[a <= -1.5] = 2
    out[a <= -2.0] = 1
    out[a <= -2.5] = 0

    cls[mask] = out
    return xr.DataArray(cls, coords=da.coords, dims=da.dims)

# Discrete palette (no white)
labels = ["Exceptional", "Extreme", "Severe", "Moderate", "Mild", "Near-normal"]
colors = ["#542788", "#b2182b", "#ef8a62", "#fdae61", "#a6d96a", "#1a9850"]
cmap = ListedColormap(colors)
cmap.set_bad("white")  # NaNs appear white

# Class edges centered on integers 0..5
class_edges = np.arange(-0.5, 6.5, 1.0)  # [-0.5, 0.5, 1.5, ..., 5.5]
norm = BoundaryNorm(class_edges, cmap.N, clip=False)
ticks = np.arange(0, 6, 1)

# -------------------------
# Plot a single year for a single file
# -------------------------
def plot_year_monthly_classes(nc_path: Path, fig_title: str, out_tmpl: str, year: int):
    ds = xr.open_dataset(nc_path)
    vname = "VDI" if "VDI" in ds else list(ds.data_vars)[0]
    da = ds[vname].sortby(["lat", "lon"])

    if "time" not in da.dims:
        print(f"{nc_path.name}: no 'time' dimension. Skipping.")
        return

    sub = da.sel(time=da["time"].dt.year == year)
    if sub.sizes.get("time", 0) == 0:
        print(f"{nc_path.name}: no data for {year}. Skipping.")
        return

    # Prepare 12 monthly slices (average within month if multiple timesteps)
    months = np.arange(1, 13, dtype=int)
    panels = []
    for m in months:
        mm = sub.sel(time=sub["time"].dt.month == m)
        if mm.sizes["time"] > 0:
            panels.append(mm.mean("time", skipna=True))
        else:
            panels.append(da.isel(time=0, drop=True) * np.nan)
    cube = xr.concat(panels, dim=("month")).assign_coords(month=("month", months))

    # Classify to integers 0..5 (keep NaNs)
    cube_cls = classify_to_int(cube)

    # Extent for imshow
    lat = cube["lat"].values
    lon = cube["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]

    # Figure with 12 panels
    fig, axes = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True, facecolor="white")
    axes = axes.ravel()
    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    for i, m in enumerate(months):
        ax = axes[i]
        panel = cube_cls.sel(month=m)
        ax.imshow(
            panel.values,
            origin="lower",
            extent=extent,
            cmap=cmap,
            norm=norm,
            interpolation="nearest",
        )
        gdf.boundary.plot(ax=ax, color="k", linewidth=0.6)
        ax.set_title(month_names[i], fontsize=11, color="black")
        ax.set_axis_off()
        ax.set_facecolor("white")

    # Single colorbar on the right with class labels at integer ticks
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), orientation="vertical", fraction=0.03, pad=0.02)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels(labels)
    cbar.ax.tick_params(labelsize=9)
    cbar.set_label("")  # no title text

    fig.suptitle(f"{fig_title} — {year} (Drought Classes)", fontsize=14, color="black")

    out_png = VDI_DIR / out_tmpl.format(year=year)
    fig.savefig(out_png, dpi=300, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_png)

# -------------------------
# Generate all figures
# -------------------------
for year in YEARS:
    for nc_path, title, tmpl in FILES:
        plot_year_monthly_classes(nc_path, title, tmpl, year)


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_allbiomes_2002_classes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_forest_2002_classes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_shrub_grass_2002_classes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_cropland_2002_classes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_allbiomes_2005_classes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_forest_2005_classes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_shrub_grass_2005_classes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_cropland_2005_classes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_allbiomes_2009_classes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_forest_2009_classes.png
Saved: C:\Drought\Regridding and data

In [35]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap, ScalarMappable

# -------------------------
# Paths
# -------------------------
VDI_DIR   = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
INDIA_SHP = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

# VDI products (final)
VDI_FILES = [
    (VDI_DIR / "VDI_allbiomes.nc",    "VDI (All Biomes)",           "CORR_{drv}_vs_VDI_allbiomes.png"),
    (VDI_DIR / "VDI_forest.nc",       "VDI (Forest)",               "CORR_{drv}_vs_VDI_forest.png"),
    (VDI_DIR / "VDI_shrub_grass.nc",  "VDI (Shrub + Grasslands)",   "CORR_{drv}_vs_VDI_shrub_grass.png"),
    (VDI_DIR / "VDI_cropland.nc",     "VDI (Cropland)",             "CORR_{drv}_vs_VDI_cropland.png"),
]

# Drivers (Z-scores, monthly) in the same VDI directory
DRIVERS = [
    (VDI_DIR / "ESI_dryz.nc",  "ESI_dryz"),
    (VDI_DIR / "SMroot_dryz.nc", "SMroot_dryz"),
    (VDI_DIR / "SIF_dryz.nc",  "SIF_dryz"),
    (VDI_DIR / "TVDI_Z_monthly_India_0p05deg_2001-2023.nc", "TVDI_Z"),
    (VDI_DIR / "VOD_dryz.nc",  "VOD_dryz"),
]

# -------------------------
# Helpers
# -------------------------
def open_var(path: Path, preferred_var: str = None) -> xr.DataArray:
    ds = xr.open_dataset(path)
    if preferred_var and preferred_var in ds:
        da = ds[preferred_var]
    else:
        da = ds[list(ds.data_vars)[0]]
    # standardize dims to (time, lat, lon)
    ren = {}
    for d in da.dims:
        dlow = d.lower()
        if dlow in ("y","latitude","lat"):  ren[d] = "lat"
        if dlow in ("x","longitude","lon"): ren[d] = "lon"
    if ren: da = da.rename(ren)
    order = [d for d in ("time","lat","lon") if d in da.dims]
    da = da.transpose(*order, ...)
    return da

def to_month_end(da: xr.DataArray) -> xr.DataArray:
    if "time" not in da.dims:
        return da
    t = pd.to_datetime(da.time.values)
    t_end = pd.to_datetime([pd.Timestamp(tt).to_period("M").asfreq("M").to_timestamp() for tt in t])
    return da.assign_coords(time=t_end)

def align_to_ref_grid(da: xr.DataArray, ref_lat: xr.DataArray, ref_lon: xr.DataArray) -> xr.DataArray:
    # sort and interp to match the reference (nearest to preserve z-scale)
    da_sorted = da.sortby([c for c in ["lat","lon"] if c in da.dims])
    ref_lat_sorted = ref_lat.sortby(ref_lat) if hasattr(ref_lat, "sortby") else ref_lat
    ref_lon_sorted = ref_lon.sortby(ref_lon) if hasattr(ref_lon, "sortby") else ref_lon
    out = da_sorted.interp(lat=ref_lat_sorted, lon=ref_lon_sorted,
                           method="nearest", kwargs={"fill_value": np.nan})
    return out

def corr_by_month(A: xr.DataArray, B: xr.DataArray) -> xr.DataArray:
    """
    For each calendar month m, compute Pearson r between A and B across years:
      r_m(lat,lon) = corr( A[time in month m], B[time in month m] )
    Returns DataArray with dims (month, lat, lon), month=1..12.
    """
    if "time" not in A.dims or "time" not in B.dims:
        raise ValueError("Both inputs must have 'time' dimension.")
    # monthly groups
    months = np.arange(1, 13, dtype=int)
    out_list = []
    for m in months:
        a_m = A.sel(time=A.time.dt.month == m)
        b_m = B.sel(time=B.time.dt.month == m)
        # Require at least 3 time samples to compute correlation
        if a_m.sizes.get("time", 0) >= 3 and b_m.sizes.get("time", 0) >= 3:
            r = xr.corr(a_m, b_m, dim="time")
        else:
            # make a NaN field on same grid
            r = A.isel(time=0, drop=True) * np.nan
        out_list.append(r)
    R = xr.concat(out_list, dim="month")
    R = R.assign_coords(month=("month", months))
    return R

# -------------------------
# Load India boundary
# -------------------------
gdf = gpd.read_file(INDIA_SHP)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS; please define it (EPSG:4326).")
gdf = gdf.to_crs("EPSG:4326")

# -------------------------
# Load drivers first, prep time to month-end
# (We'll align each driver to the VDI grid inside the loop.)
# -------------------------
drivers = {}
for p, vname in DRIVERS:
    da = open_var(p, vname)
    da = to_month_end(da)
    drivers[vname] = da

# -------------------------
# Plotting helper
# -------------------------
def plot_corr_12panels(R: xr.DataArray, title: str, out_path: Path):
    # Ensure (month, lat, lon)
    R = R.transpose("month","lat","lon")
    lat = R["lat"].values
    lon = R["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]

    cmap = get_cmap("rainbow").copy()
    cmap.set_bad("white")
    norm = Normalize(vmin=-1.0, vmax=1.0)

    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    fig, axes = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True, facecolor="white")
    axes = axes.ravel()

    for i in range(12):
        ax = axes[i]
        frame = R.sel(month=i+1)
        ax.imshow(frame.values, origin="lower", extent=extent, cmap=cmap, norm=norm, interpolation="nearest")
        gdf.boundary.plot(ax=ax, color="k", linewidth=0.6)
        ax.set_title(month_names[i], fontsize=11, color="black")
        ax.set_axis_off()
        ax.set_facecolor("white")

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), orientation="vertical", fraction=0.03, pad=0.02)
    cbar.set_label("")  # no label text
    fig.suptitle(title, fontsize=14, color="black")

    fig.savefig(out_path, dpi=300, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)

# -------------------------
# Main loop: for each driver × each VDI product
# -------------------------
for vdi_path, vdi_title, out_tmpl in VDI_FILES:
    # Load VDI
    ds_vdi = xr.open_dataset(vdi_path)
    vname_vdi = "VDI" if "VDI" in ds_vdi else list(ds_vdi.data_vars)[0]
    VDI = ds_vdi[vname_vdi]
    VDI = VDI.sortby(["lat","lon"])
    VDI = to_month_end(VDI)

    # Reference grid
    ref_lat = VDI["lat"]
    ref_lon = VDI["lon"]

    # Time axis for intersection
    t_vdi = set(VDI.time.values) if "time" in VDI.dims else set()

    for d_path, d_name in DRIVERS:
        Drv = drivers[d_name]

        # Align to VDI time: intersection
        if "time" not in Drv.dims:
            print(f"Driver {d_name} has no time dimension; skipping.")
            continue
        common_time = sorted(t_vdi & set(Drv.time.values))
        if len(common_time) < 3*12:  # at least 3 years worth for robust monthly r
            print(f"Warning: {d_name} ∩ VDI has limited overlap ({len(common_time)} months). Proceeding anyway.")
        VDIc = VDI.sel(time=common_time)
        DRVc = Drv.sel(time=common_time)

        # Align to VDI grid (nearest)
        DRVc = align_to_ref_grid(DRVc, ref_lat, ref_lon)

        # Compute monthly correlations
        R = corr_by_month(DRVc, VDIc)  # per-month r in [-1,1]

        # Plot & save
        out_png = VDI_DIR / out_tmpl.format(drv=d_name)
        plot_corr_12panels(R, f"Correlation: {d_name} vs {vdi_title}", out_png)


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1922896000.py:138: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORR_ESI_dryz_vs_VDI_allbiomes.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1922896000.py:138: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORR_SMroot_dryz_vs_VDI_allbiomes.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1922896000.py:138: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


...

Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORR_SIF_dryz_vs_VDI_cropland.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1922896000.py:138: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORR_TVDI_Z_vs_VDI_cropland.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1922896000.py:138: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORR_VOD_dryz_vs_VDI_cropland.png


In [37]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from itertools import combinations
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap, ScalarMappable

# -------------------------------------------------
# Paths
# -------------------------------------------------
VDI_DIR   = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
LULC_DIR  = Path(r"C:\Drought\Land Cover")
INDIA_SHP = LULC_DIR / "India_with_jk.shp"

# Driver files: (path, variable name in file)
DRIVERS = [
    (VDI_DIR / "ESI_dryz.nc",                                 "ESI_dryz"),
    (VDI_DIR / "SMroot_dryz.nc",                              "SMroot_dryz"),
    (VDI_DIR / "SIF_dryz.nc",                                 "SIF_dryz"),
    (VDI_DIR / "TVDI_Z_monthly_India_0p05deg_2001-2023.nc",   "TVDI_Z"),
    (VDI_DIR / "VOD_dryz.nc",                                 "VOD_dryz"),
]

# Lags (months) to test
LAGS = [0, 1, 2, 3]

# -------------------------------------------------
# Helpers: IO & alignment
# -------------------------------------------------
def open_var(path: Path, preferred_var: str) -> xr.DataArray:
    ds = xr.open_dataset(path)
    da = ds[preferred_var] if preferred_var in ds else ds[list(ds.data_vars)[0]]
    # standardize dims to (time, lat, lon)
    ren = {}
    for d in da.dims:
        dlow = d.lower()
        if dlow in ("y", "latitude", "lat"):  ren[d] = "lat"
        if dlow in ("x", "longitude", "lon"): ren[d] = "lon"
    if ren:
        da = da.rename(ren)
    # sort & transpose
    order = [d for d in ("time","lat","lon") if d in da.dims]
    da = da.transpose(*order, ...)
    return da.sortby([c for c in ["lat","lon"] if c in da.dims])

def to_month_end(da: xr.DataArray) -> xr.DataArray:
    if "time" not in da.dims:
        return da
    t = pd.to_datetime(da.time.values)
    t_end = pd.to_datetime([pd.Timestamp(tt).to_period("M").asfreq("M").to_timestamp() for tt in t])
    return da.assign_coords(time=t_end)

def align_to_ref_grid(da: xr.DataArray, ref_lat: xr.DataArray, ref_lon: xr.DataArray) -> xr.DataArray:
    # nearest-neighbor to preserve anomalies and avoid smoothing
    da_sorted = da.sortby([c for c in ["lat","lon"] if c in da.dims])
    out = da_sorted.interp(lat=ref_lat.sortby(ref_lat), lon=ref_lon.sortby(ref_lon),
                           method="nearest", kwargs={"fill_value": np.nan})
    return out

# -------------------------------------------------
# Load India boundary
# -------------------------------------------------
gdf = gpd.read_file(INDIA_SHP)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS; please define it (should be EPSG:4326).")
gdf = gdf.to_crs("EPSG:4326")

# -------------------------------------------------
# Load all drivers; pick a reference grid (ESI)
# -------------------------------------------------
drivers = {}
for p, v in DRIVERS:
    da = open_var(p, v)
    da = to_month_end(da)
    drivers[v] = da

# Choose reference grid (ESI)
ref = drivers["ESI_dryz"]
ref_lat, ref_lon = ref["lat"], ref["lon"]

# Align all drivers to ref grid and intersect common time across ALL drivers
for v in list(drivers.keys()):
    drivers[v] = align_to_ref_grid(drivers[v], ref_lat, ref_lon)

common_time = set(drivers[list(drivers.keys())[0]].time.values)
for v in drivers.values():
    common_time &= set(v.time.values)
common_time = np.array(sorted(common_time))
for k in list(drivers.keys()):
    drivers[k] = drivers[k].sel(time=common_time)

# -------------------------------------------------
# Build biome masks (2D), aligned to ref grid
# -------------------------------------------------
def open_fraction(path: Path) -> xr.DataArray:
    ds = xr.open_dataset(path)
    var = "fraction" if "fraction" in ds else list(ds.data_vars)[0]
    da = ds[var]
    ren = {}
    for d in da.dims:
        dlow = d.lower()
        if dlow in ("y","latitude","lat"):  ren[d] = "lat"
        if dlow in ("x","longitude","lon"): ren[d] = "lon"
    if ren:
        da = da.rename(ren)
    da = da.transpose("lat","lon")
    da = align_to_ref_grid(da, ref_lat, ref_lon)
    return da.fillna(0.0).astype("float32")

frac_forest = open_fraction(LULC_DIR / "lulc_forest_frac_0p05deg.nc")
frac_shrub  = open_fraction(LULC_DIR / "lulc_shrub_grass_frac_0p05deg.nc")
frac_crop   = open_fraction(LULC_DIR / "lulc_cropland_frac_0p05deg.nc")

# Dominant biome code: 1=forest, 2=shrub/grass, 3=cropland; NaN where none
stacked = xr.concat([frac_forest, frac_shrub, frac_crop], dim="biome")  # 0,1,2
dom_idx = stacked.argmax("biome")
none = (stacked.sum("biome") <= 0)

biome_code = xr.full_like(frac_forest, np.nan, dtype="float32")
biome_code = xr.where(dom_idx==0, 1.0, biome_code)
biome_code = xr.where(dom_idx==1, 2.0, biome_code)
biome_code = xr.where(dom_idx==2, 3.0, biome_code)
biome_code = biome_code.where(~none)

MASKS = {
    "AllIndia": xr.ones_like(frac_forest, dtype="float32"),
    "Forest":   xr.where(biome_code==1.0, 1.0, 0.0),
    "ShrubGrass": xr.where(biome_code==2.0, 1.0, 0.0),
    "Cropland": xr.where(biome_code==3.0, 1.0, 0.0),
}

# -------------------------------------------------
# Correlation with lag:
# r_m(lat,lon) = corr( A[t in month m], B[t+lag in month m] )
# Implemented by shifting B backward by 'lag' so B_shift(t) = B(t+lag).
# Then intersect time and correlate across years per month.
# -------------------------------------------------
def corr_by_month_lag(A: xr.DataArray, B: xr.DataArray, lag: int) -> xr.DataArray:
    if lag < 0:
        raise ValueError("Lag must be >= 0.")
    if "time" not in A.dims or "time" not in B.dims:
        raise ValueError("Both inputs must have 'time' dimension.")
    B_shift = B.shift(time=-lag) if lag > 0 else B

    # Keep only overlapping (non-NaN) time points after shift
    common = set(A.time.values) & set(B_shift.time.values)
    if not common:
        # return NaNs on same grid & 12 months
        tmpl = A.isel(time=0, drop=True) * np.nan
        return xr.concat([tmpl]*12, dim="month").assign_coords(month=("month", np.arange(1,13)))
    common = np.array(sorted(common))
    A2 = A.sel(time=common)
    B2 = B_shift.sel(time=common)

    # Per-month correlation
    outs = []
    months = np.arange(1, 13, dtype=int)
    for m in months:
        a_m = A2.sel(time=A2.time.dt.month == m)
        b_m = B2.sel(time=B2.time.dt.month == m)
        if a_m.sizes.get("time", 0) >= 3 and b_m.sizes.get("time", 0) >= 3:
            r = xr.corr(a_m, b_m, dim="time")
        else:
            r = A2.isel(time=0, drop=True) * np.nan
        outs.append(r)
    R = xr.concat(outs, dim="month").assign_coords(month=("month", months))
    return R

# -------------------------------------------------
# Plotting helper (12 panels)
# -------------------------------------------------
def plot_12panels(R: xr.DataArray, title: str, out_path: Path):
    R = R.transpose("month","lat","lon")
    lat = R["lat"].values
    lon = R["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]

    cmap = get_cmap("rainbow").copy()
    cmap.set_bad("white")
    norm = Normalize(vmin=-1.0, vmax=1.0)

    fig, axes = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True, facecolor="white")
    axes = axes.ravel()
    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    for i in range(12):
        ax = axes[i]
        frame = R.sel(month=i+1)
        ax.imshow(frame.values, origin="lower", extent=extent, cmap=cmap, norm=norm, interpolation="nearest")
        gdf.boundary.plot(ax=ax, color="k", linewidth=0.6)
        ax.set_title(month_names[i], fontsize=11, color="black")
        ax.set_axis_off()
        ax.set_facecolor("white")

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), orientation="vertical", fraction=0.03, pad=0.02)
    cbar.set_label("")  # no label text
    fig.suptitle(title, fontsize=14, color="black")

    fig.savefig(out_path, dpi=300, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)

# -------------------------------------------------
# Driver pairs × lags × biome masks
# -------------------------------------------------
driver_names = [v for _, v in DRIVERS]
pairs = list(combinations(driver_names, 2))  # all unique pairs (5 choose 2 = 10)

for drvA, drvB in pairs:
    A = drivers[drvA]
    B = drivers[drvB]
    for lag in LAGS:
        # compute correlation (no biome mask yet)
        R = corr_by_month_lag(A, B, lag=lag)  # (month, lat, lon)
        for biome_label, mask in MASKS.items():
            # clip to biome by masking outside (keep inside as is)
            R_masked = R.where(mask == 1.0)
            out_png = VDI_DIR / f"CORRPAIR_{drvA}_vs_{drvB}_lag{lag}_{biome_label}.png"
            title = f"Corr ({drvA} vs {drvB}) — Lag {lag} mo — {biome_label}"
            plot_12panels(R_masked, title, out_png)


<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2062095551.py:1: SyntaxWarning: invalid escape sequence '\D'
  """
C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2062095551.py:208: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORRPAIR_ESI_dryz_vs_SMroot_dryz_lag0_AllIndia.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2062095551.py:208: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORRPAIR_ESI_dryz_vs_SMroot_dryz_lag0_Forest.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2062095551.py:208: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


...

Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORRPAIR_TVDI_Z_vs_VOD_dryz_lag3_Forest.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2062095551.py:208: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORRPAIR_TVDI_Z_vs_VOD_dryz_lag3_ShrubGrass.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2062095551.py:208: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORRPAIR_TVDI_Z_vs_VOD_dryz_lag3_Cropland.png


In [39]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, ListedColormap
from matplotlib.cm import get_cmap, ScalarMappable
from scipy.stats import linregress

# -------------------------
# Paths
# -------------------------
VDI_DIR   = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
INDIA_SHP = Path(r"C:\Drought\Land Cover\India_with_jk.shp")

FILES = {
    "AllIndia":     VDI_DIR / "VDI_allbiomes.nc",
    "Forest":       VDI_DIR / "VDI_forest.nc",
    "ShrubGrass":   VDI_DIR / "VDI_shrub_grass.nc",
    "Cropland":     VDI_DIR / "VDI_cropland.nc",
}

# -------------------------
# Load India outline (for plotting overlays)
# -------------------------
gdf = gpd.read_file(INDIA_SHP)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS; please set it (EPSG:4326).")
gdf = gdf.to_crs("EPSG:4326")

# -------------------------
# Drought-class thresholds and labels
# -------------------------
def classify_to_int(da: xr.DataArray) -> xr.DataArray:
    """Return integer classes 0..5 with NaNs preserved."""
    arr = da.values
    out = np.full(arr.shape, np.nan, dtype="float32")
    m = ~np.isnan(arr)
    a = arr[m]
    cls = np.full(a.shape, 5, dtype="int16")  # default Near-normal
    cls[a <= -0.5] = 4
    cls[a <= -1.0] = 3
    cls[a <= -1.5] = 2
    cls[a <= -2.0] = 1
    cls[a <= -2.5] = 0
    out[m] = cls
    return xr.DataArray(out, coords=da.coords, dims=da.dims)

CLASS_ORDER = [0,1,2,3,4,5]
CLASS_LABEL = {
    0: "Exceptional",
    1: "Extreme",
    2: "Severe",
    3: "Moderate",
    4: "Mild",
    5: "Near-normal",
}

# -------------------------
# Utilities
# -------------------------
def open_vdi(path: Path) -> xr.DataArray:
    ds = xr.open_dataset(path)
    vname = "VDI" if "VDI" in ds else list(ds.data_vars)[0]
    da = ds[vname]
    # unify dims, sort
    ren = {}
    for d in da.dims:
        if d.lower() in ("y","latitude"): ren[d] = "lat"
        if d.lower() in ("x","longitude"): ren[d] = "lon"
    if ren: da = da.rename(ren)
    da = da.transpose(*[d for d in ("time","lat","lon") if d in da.dims], ...)
    # ensure time is month-end
    if "time" in da.dims:
        t = pd.to_datetime(da.time.values)
        t_end = pd.to_datetime([pd.Timestamp(tt).to_period("M").asfreq("M").to_timestamp() for tt in t])
        da = da.assign_coords(time=t_end)
    return da.sortby(["lat","lon"])

def area_weights(lat: np.ndarray) -> np.ndarray:
    """Cosine(lat) weights for simple area-average in geographic grid (normalized later)."""
    w = np.cos(np.deg2rad(lat))
    return w / np.nanmean(w)

def area_mean(da: xr.DataArray) -> xr.DataArray:
    """Area-weighted mean over lat,lon."""
    w = xr.DataArray(area_weights(da["lat"].values), coords={"lat": da["lat"]}, dims=("lat",))
    num = (da * w).mean(dim=("lon"))  # weight only lat (lon cell width ~const at small domain)
    return num.mean(dim=("lat"))

def linear_trend_along_time(da: xr.DataArray) -> xr.DataArray:
    """
    Compute slope (per year) at each grid cell using linear regression over time.
    Assumes 'time' coord spans multiple years of monthly or seasonal data.
    """
    if "time" not in da.dims:
        raise ValueError("Input must have a 'time' dimension.")
    # x as fractional year
    years = pd.to_datetime(da.time.values).year
    x = xr.DataArray(years.astype("float32"), coords={"time": da.time}, dims=("time",))
    # demean x to reduce numerical issues
    x0 = x - x.mean("time")
    # slope = cov(x,y)/var(x)
    y = da
    # mask NaN time steps consistently
    valid = np.isfinite(y).all(dim=set(y.dims) - {"time"}) if False else np.isfinite(y)
    # compute components
    xy = (x0 * y).mean("time", skipna=True)
    xx = (x0 * x0).mean("time", skipna=True)
    slope = xy / xx
    # Convert from "per year-unit of x0" to per year: x0 is already in years diff, so slope is per year.
    return slope

# -------------------------
# 1) Classification-wise monthly climatology (per biome) → CSV
# -------------------------
records = []
for biome_name, nc_path in FILES.items():
    da = open_vdi(nc_path)  # (time, lat, lon)
    if "time" not in da.dims:
        print(f"[WARN] {nc_path.name} has no time dimension; skipping CSV step.")
        continue
    # Monthly climatology
    monthly = da.groupby("time.month").mean("time", skipna=True)  # (month, lat, lon)
    # Classify each month’s field into classes
    cls = classify_to_int(monthly)  # (month, lat, lon), values 0..5 or NaN
    # For each month & class: average VDI over pixels of that class, and area fraction
    for m in range(1, 13):
        v = monthly.sel(month=m)
        c = cls.sel(month=m)
        for k in CLASS_ORDER:
            mask = (c == k)
            pix_count = int(mask.sum().values) if np.isfinite(mask).any() else 0
            if pix_count == 0:
                mean_vdi = np.nan
                frac = 0.0
            else:
                mean_vdi = float(v.where(mask).mean().values)
                total_pix = int(np.isfinite(v).sum().values)
                frac = float(pix_count / total_pix) if total_pix > 0 else 0.0
            records.append({
                "Biome": biome_name,
                "Month": m,
                "ClassCode": k,
                "ClassName": CLASS_LABEL[k],
                "MeanVDI": mean_vdi,
                "PixelFraction": frac,
                "PixelCount": pix_count,
            })

# Save CSV
df = pd.DataFrame.from_records(records)
csv_path = VDI_DIR / "VDI_monthly_climatology_by_class_and_biome.csv"
df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

# -------------------------
# 2) Monthly linear trends (12 panels, AllIndia map)
# -------------------------
def plot_12_month_trend(da: xr.DataArray, title: str, out_png: Path):
    # Build per-month series, get slope per month
    slopes = []
    for m in range(1, 13):
        sub = da.sel(time=da.time.dt.month == m)
        if sub.sizes["time"] >= 3:
            slopes.append(linear_trend_along_time(sub))
        else:
            slopes.append(da.isel(time=0, drop=True) * np.nan)
    trend = xr.concat(slopes, dim="month").assign_coords(month=("month", np.arange(1,13)))

    # Plot
    lat = trend["lat"].values; lon = trend["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]
    cmap = get_cmap("rainbow").copy(); cmap.set_bad("white")
    # Choose symmetric scale around 0 for slopes; robust limits
    vmin, vmax = np.nanpercentile(trend.values, [2,98])
    val = max(abs(vmin), abs(vmax)); vmin, vmax = -val, val
    norm = Normalize(vmin=vmin, vmax=vmax)

    fig, axes = plt.subplots(3,4, figsize=(16,10), constrained_layout=True, facecolor="white")
    axes = axes.ravel()
    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    for i in range(12):
        ax = axes[i]
        ax.imshow(trend.sel(month=i+1).values, origin="lower", extent=extent, cmap=cmap, norm=norm, interpolation="nearest")
        gdf.boundary.plot(ax=ax, color="k", linewidth=0.6)
        ax.set_title(month_names[i], fontsize=11, color="black")
        ax.set_axis_off(); ax.set_facecolor("white")
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), orientation="vertical", fraction=0.03, pad=0.02)
    cbar.set_label("")  # no text
    fig.suptitle(title, fontsize=14, color="black")
    fig.savefig(out_png, dpi=300, facecolor="white", bbox_inches="tight"); plt.close(fig)
    print("Saved:", out_png)

all_da = open_vdi(FILES["AllIndia"])
if "time" in all_da.dims:
    plot_12_month_trend(all_da, "VDI Monthly Linear Trend (slope per year) — All Biomes",
                        VDI_DIR / "VDI_monthly_trend_allbiomes.png")

# -------------------------
# 3) Seasonal climatology & trends (DJF, MAM, JJAS, ON) for all and per-biome
# -------------------------
SEASONS = {
    "DJF":  [12, 1, 2],
    "MAM":  [3, 4, 5],
    "JJAS": [6, 7, 8, 9],
    "ON":   [10, 11],   # per your note "ON"
}

def season_mean_series(da: xr.DataArray, months: list) -> xr.DataArray:
    """Return a seasonal time series (one value per year) averaging given months within each year."""
    if "time" not in da.dims: return da * np.nan
    df = da.to_dataframe(name="v").reset_index()
    df["year"]  = pd.to_datetime(df["time"]).dt.year
    df["month"] = pd.to_datetime(df["time"]).dt.month
    df = df[df["month"].isin(months)]
    # mean within year/month-set
    grp = df.groupby(["year","lat","lon"])["v"].mean().reset_index()
    # back to DataArray: dims (year, lat, lon)
    years = np.sort(grp["year"].unique())
    lat   = np.sort(df["lat"].unique()); lon = np.sort(df["lon"].unique())
    arr = np.full((len(years), len(lat), len(lon)), np.nan, dtype="float32")
    # pivot efficiently
    piv = grp.pivot_table(index=["year","lat","lon"], values="v").reset_index()
    y_index = {y:i for i,y in enumerate(years)}
    lat_index = {la:i for i,la in enumerate(lat)}
    lon_index = {lo:i for i,lo in enumerate(lon)}
    for _, r in piv.iterrows():
        arr[y_index[r["year"]], lat_index[r["lat"]], lon_index[r["lon"]]] = float(r["v"])
    out = xr.DataArray(arr, coords={"year":years, "lat":lat, "lon":lon}, dims=("year","lat","lon"))
    return out

def plot_season_map(da_map: xr.DataArray, title: str, out_png: Path, symmetric=False):
    lat = da_map["lat"].values; lon = da_map["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]
    cmap = get_cmap("rainbow").copy(); cmap.set_bad("white")
    if symmetric:
        vmin, vmax = np.nanpercentile(da_map.values, [2,98])
        v = max(abs(vmin), abs(vmax)); vmin, vmax = -v, v
    else:
        vmin, vmax = np.nanpercentile(da_map.values, [2,98])
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin==vmax:
            vmin, vmax = -2.5, 2.5
    norm = Normalize(vmin=vmin, vmax=vmax)

    fig, ax = plt.subplots(figsize=(7.5,6), facecolor="white", constrained_layout=True)
    ax.imshow(da_map.values, origin="lower", extent=extent, cmap=cmap, norm=norm, interpolation="nearest")
    gdf.boundary.plot(ax=ax, color="k", linewidth=0.7)
    ax.set_axis_off(); ax.set_facecolor("white"); fig.suptitle(title, fontsize=13, color="black")
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation="vertical", fraction=0.035, pad=0.02); cbar.set_label("")
    fig.savefig(out_png, dpi=300, facecolor="white", bbox_inches="tight"); plt.close(fig)
    print("Saved:", out_png)

for biome_name, nc_path in FILES.items():
    da = open_vdi(nc_path)
    if "time" not in da.dims: 
        print(f"[WARN] {nc_path.name} has no time; skipping seasonal products.")
        continue
    for sname, months in SEASONS.items():
        # Seasonal climatology (mean over years)
        clim = da.sel(time=da.time.dt.month.isin(months)).groupby("time.year").mean("time").mean("year")
        plot_season_map(clim, f"VDI Seasonal Climatology — {sname} — {biome_name}",
                        VDI_DIR / f"VDI_{biome_name}_{sname}_climatology.png", symmetric=False)
        # Seasonal trend: slope per year from seasonal series
        seas_ts = season_mean_series(da, months)    # (year, lat, lon)
        if seas_ts.sizes.get("year",0) >= 3:
            # compute slope along 'year'
            x = xr.DataArray(seas_ts["year"].astype("float32"), coords={"year":seas_ts["year"]}, dims=("year",))
            x0 = x - x.mean("year")
            slope = ( (x0*seas_ts).mean("year") / (x0*x0).mean("year") )
            slope = slope.transpose("lat","lon")
        else:
            slope = da.isel(time=0, drop=True) * np.nan
        plot_season_map(slope, f"VDI Seasonal Trend (slope/yr) — {sname} — {biome_name}",
                        VDI_DIR / f"VDI_{biome_name}_{sname}_trend.png", symmetric=True)

# -------------------------
# 4) Seasonal “stripes” (India-average) per biome
# -------------------------
def season_avg_series_scalar(da: xr.DataArray, months: list) -> pd.Series:
    """Return seasonal India-average VDI per year (area-weighted)."""
    seas = season_mean_series(da, months)  # (year, lat, lon)
    # area mean
    w = xr.DataArray(np.cos(np.deg2rad(seas["lat"].values)),
                     coords={"lat":seas["lat"]}, dims=("lat",))
    m = (seas * w).mean("lon").mean("lat")
    return pd.Series(m.values, index=pd.Index(seas["year"].values, name="year"))

def plot_stripes(series: pd.Series, title: str, out_png: Path, vmin=-2.5, vmax=2.5):
    """Single-row stripes plot (green→red; wet→dry)."""
    years = series.index.values
    data = (series - 0.0).values  # already Z-like
    # prepare an array shape (1, n_years)
    stripe = data.reshape(1, -1)
    cmap = plt.get_cmap("RdYlGn_r").copy()  # green=wet -> red=dry
    cmap.set_bad("white")
    norm = Normalize(vmin=vmin, vmax=vmax)

    fig, ax = plt.subplots(figsize=(12, 1.2), facecolor="white", constrained_layout=True)
    im = ax.imshow(stripe, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_axis_off(); ax.set_facecolor("white")
    fig.suptitle(title, fontsize=12, color="black")
    # tiny colorbar
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation="horizontal", fraction=0.15, pad=0.2)
    cbar.set_label(""); cbar.ax.tick_params(labelsize=8)
    fig.savefig(out_png, dpi=300, facecolor="white", bbox_inches="tight"); plt.close(fig)
    print("Saved:", out_png)

# One “4-row panel” (one row per season) per biome + AllIndia
def plot_stripes_4seasons(da: xr.DataArray, biome_name: str):
    rows = []
    titles = []
    years_union = None
    for sname, months in SEASONS.items():
        s = season_avg_series_scalar(da, months)  # pd.Series per year
        rows.append(s)
        titles.append(sname)
        years_union = s.index if years_union is None else years_union.union(s.index)
    # Align series by full year range
    all_years = np.array(sorted(years_union.values))
    mat = []
    for s in rows:
        mat.append(pd.Series(s).reindex(all_years).values)
    M = np.vstack(mat)  # shape (4, n_years)
    cmap = plt.get_cmap("RdYlGn_r").copy()
    cmap.set_bad("white")
    vmin, vmax = -2.5, 2.5

    fig, ax = plt.subplots(figsize=(12, 5), facecolor="white", constrained_layout=True)
    im = ax.imshow(M, aspect="auto", cmap=cmap, norm=Normalize(vmin=vmin, vmax=vmax), interpolation="nearest")
    # Row labels on the left
    ax.set_yticks(np.arange(len(SEASONS))); ax.set_yticklabels(list(SEASONS.keys()))
    ax.set_xticks([])  # clean look; optional to add years as sparse ticks
    ax.set_facecolor("white")
    fig.suptitle(f"India-average VDI seasonal stripes — {biome_name}", fontsize=13, color="black")
    sm = ScalarMappable(norm=Normalize(vmin=vmin,vmax=vmax), cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation="horizontal", fraction=0.06, pad=0.08); cbar.set_label("")
    fig.savefig(VDI_DIR / f"VDI_{biome_name}_seasonal_stripes.png", dpi=300, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    print("Saved:", VDI_DIR / f"VDI_{biome_name}_seasonal_stripes.png")

for biome_name, nc_path in FILES.items():
    da = open_vdi(nc_path)
    if "time" in da.dims:
        plot_stripes_4seasons(da, biome_name)


<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2769647631.py:1: SyntaxWarning: invalid escape sequence '\D'
  """


Saved CSV: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_monthly_climatology_by_class_and_biome.csv


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2769647631.py:190: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy(); cmap.set_bad("white")


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_monthly_trend_allbiomes.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2769647631.py:253: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy(); cmap.set_bad("white")


...

Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_Cropland_JJAS_trend.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2769647631.py:253: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy(); cmap.set_bad("white")


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_Cropland_ON_climatology.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2769647631.py:253: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy(); cmap.set_bad("white")


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_Cropland_ON_trend.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_AllIndia_seasonal_stripes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_Forest_seasonal_stripes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_ShrubGrass_seasonal_stripes.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_Cropland_seasonal_stripes.png


In [41]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable

# -------------------------
# Paths (adjust if needed)
# -------------------------
VDI_DIR = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
INDIA_SHP = Path(r"C:\Drought\Land Cover\India_with_jk.shp")

FILES = [
    (VDI_DIR / "VDI_allbiomes.nc",   "VDI (All Biomes)",           "VDI_allbiomes_{year}_classes_POS.png"),
    (VDI_DIR / "VDI_forest.nc",      "VDI (Forest)",               "VDI_forest_{year}_classes_POS.png"),
    (VDI_DIR / "VDI_shrub_grass.nc", "VDI (Shrub + Grasslands)",   "VDI_shrub_grass_{year}_classes_POS.png"),
    (VDI_DIR / "VDI_cropland.nc",    "VDI (Cropland)",             "VDI_cropland_{year}_classes_POS.png"),
]

YEARS = [2002, 2005, 2009, 2015, 2019]  # change if you want different years

# -------------------------
# Load India outline
# -------------------------
gdf = gpd.read_file(INDIA_SHP)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS; please set it (EPSG:4326).")
gdf = gdf.to_crs("EPSG:4326")

# -------------------------
# Classification for VDI (positive = drier)
# -------------------------
def classify_to_int(da: xr.DataArray) -> xr.DataArray:
    """
    Map VDI (positive = drier) to integer classes 0..5 (NaNs preserved):

      0 Near-normal/Wet    : VDI <  +0.5
      1 Mild               : 0.5 ≤ VDI < 1.0
      2 Moderate           : 1.0 ≤ VDI < 1.5
      3 Severe             : 1.5 ≤ VDI < 2.0
      4 Extreme            : 2.0 ≤ VDI < 2.5
      5 Exceptional        : VDI ≥  +2.5
    """
    arr = da.values
    out = np.full(arr.shape, np.nan, dtype="float32")
    m = ~np.isnan(arr)
    a = arr[m]

    cls = np.zeros_like(a, dtype="int16")         # default Near-normal/Wet
    cls[(a >= 0.5)  & (a < 1.0)]  = 1            # Mild
    cls[(a >= 1.0)  & (a < 1.5)]  = 2            # Moderate
    cls[(a >= 1.5)  & (a < 2.0)]  = 3            # Severe
    cls[(a >= 2.0)  & (a < 2.5)]  = 4            # Extreme
    cls[(a >= 2.5)]               = 5            # Exceptional

    out[m] = cls
    return xr.DataArray(out, coords=da.coords, dims=da.dims)

# Labels in ascending order 0..5
CLASS_LABELS = [
    "Near-normal/Wet",  # 0
    "Mild",             # 1
    "Moderate",         # 2
    "Severe",           # 3
    "Extreme",          # 4
    "Exceptional",      # 5
]

# Discrete palette: wet→dry (green→red), no white in classes (NaNs remain white)
CLASS_COLORS = ["#1a9850", "#a6d96a", "#fdae61", "#ef8a62", "#b2182b", "#542788"]
cmap = ListedColormap(CLASS_COLORS)
cmap.set_bad("white")

# Finite class edges around integers (for imshow colorbar)
CLASS_EDGES = np.arange(-0.5, 6.5, 1.0)  # [-0.5, 0.5, ..., 5.5]
norm = BoundaryNorm(CLASS_EDGES, cmap.N, clip=False)
ticks = np.arange(0, 6, 1)

# -------------------------
# Helpers
# -------------------------
def open_vdi(nc_path: Path) -> xr.DataArray:
    ds = xr.open_dataset(nc_path)
    vname = "VDI" if "VDI" in ds else list(ds.data_vars)[0]
    da = ds[vname]
    # unify dims
    ren = {}
    for d in da.dims:
        dlow = d.lower()
        if dlow in ("y","latitude","lat"):  ren[d] = "lat"
        if dlow in ("x","longitude","lon"): ren[d] = "lon"
    if ren:
        da = da.rename(ren)
    da = da.transpose(*[d for d in ("time","lat","lon") if d in da.dims], ...)
    # standardize to month-end time
    if "time" in da.dims:
        t = pd.to_datetime(da.time.values)
        t_end = pd.to_datetime([pd.Timestamp(tt).to_period("M").asfreq("M").to_timestamp() for tt in t])
        da = da.assign_coords(time=t_end)
    return da.sortby(["lat","lon"])

def plot_year_monthly_classes(nc_path: Path, fig_title: str, out_tmpl: str, year: int):
    da = open_vdi(nc_path)

    if "time" not in da.dims:
        print(f"{nc_path.name}: no 'time' dimension. Skipping.")
        return

    sub = da.sel(time=da["time"].dt.year == year)
    if sub.sizes.get("time", 0) == 0:
        print(f"{nc_path.name}: no data for {year}. Skipping.")
        return

    # Build 12 monthly fields (mean if multiple timesteps per month)
    months = np.arange(1, 13, dtype=int)
    frames = []
    for m in months:
        mm = sub.sel(time=sub["time"].dt.month == m)
        if mm.sizes["time"] > 0:
            frames.append(mm.mean("time", skipna=True))
        else:
            frames.append(da.isel(time=0, drop=True) * np.nan)
    cube = xr.concat(frames, dim=("month")).assign_coords(month=("month", months))

    # Classify to 0..5 (keep NaNs)
    cube_cls = classify_to_int(cube)

    # Plot
    lat = cube["lat"].values
    lon = cube["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]
    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    fig, axes = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True, facecolor="white")
    axes = axes.ravel()

    for i, m in enumerate(months):
        ax = axes[i]
        panel = cube_cls.sel(month=m)
        ax.imshow(
            panel.values,
            origin="lower",
            extent=extent,
            cmap=cmap,
            norm=norm,
            interpolation="nearest",
        )
        gdf.boundary.plot(ax=ax, color="k", linewidth=0.6)
        ax.set_title(month_names[i], fontsize=11, color="black")
        ax.set_axis_off()
        ax.set_facecolor("white")

    # Single colorbar on right with class labels
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), orientation="vertical", fraction=0.03, pad=0.02)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels(CLASS_LABELS)
    cbar.ax.tick_params(labelsize=9)
    cbar.set_label("")  # no text label

    fig.suptitle(f"{fig_title} — {year} (Drought Classes, + = drier)", fontsize=14, color="black")

    out_png = VDI_DIR / out_tmpl.format(year=year)
    fig.savefig(out_png, dpi=300, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_png)

# -------------------------
# Generate all figures
# -------------------------
for year in YEARS:
    for nc_path, title, tmpl in FILES:
        plot_year_monthly_classes(nc_path, title, tmpl, year)


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_allbiomes_2002_classes_POS.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_forest_2002_classes_POS.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_shrub_grass_2002_classes_POS.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_cropland_2002_classes_POS.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_allbiomes_2005_classes_POS.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_forest_2005_classes_POS.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_shrub_grass_2005_classes_POS.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_cropland_2005_classes_POS.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_allbiomes_2009_classes_POS.png
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_forest_2009_classes_POS.p

In [43]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap, ScalarMappable

# -------------------------
# Paths
# -------------------------
VDI_DIR   = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
INDIA_SHP = Path(r"C:\Drought\Land Cover\India_with_jk.shp")

# Final VDI products (biome-specific + combined)
VDI_FILES = [
    (VDI_DIR / "VDI_allbiomes.nc",    "VDI (All Biomes)",           "CORR_{drv}_vs_VDI_allbiomes.png"),
    (VDI_DIR / "VDI_forest.nc",       "VDI (Forest)",               "CORR_{drv}_vs_VDI_forest.png"),
    (VDI_DIR / "VDI_shrub_grass.nc",  "VDI (Shrub + Grasslands)",   "CORR_{drv}_vs_VDI_shrub_grass.png"),
    (VDI_DIR / "VDI_cropland.nc",     "VDI (Cropland)",             "CORR_{drv}_vs_VDI_cropland.png"),
]

# Drivers (monthly Z-scores) in the same directory
DRIVERS = [
    (VDI_DIR / "ESI_dryz.nc",                                "ESI_dryz"),
    (VDI_DIR / "SMroot_dryz.nc",                             "SMroot_dryz"),
    (VDI_DIR / "SIF_dryz.nc",                                "SIF_dryz"),
    (VDI_DIR / "TVDI_Z_monthly_India_0p05deg_2001-2023.nc",  "TVDI_Z"),
    (VDI_DIR / "VOD_dryz.nc",                                "VOD_dryz"),
]

# -------------------------
# Helpers
# -------------------------
def open_var(path: Path, preferred_var: str | None = None) -> xr.DataArray:
    ds = xr.open_dataset(path)
    if preferred_var and preferred_var in ds:
        da = ds[preferred_var]
    else:
        da = ds[list(ds.data_vars)[0]]
    # standardize dims to (time, lat, lon)
    ren = {}
    for d in da.dims:
        dlow = d.lower()
        if dlow in ("y","latitude","lat"):  ren[d] = "lat"
        if dlow in ("x","longitude","lon"): ren[d] = "lon"
    if ren:
        da = da.rename(ren)
    order = [d for d in ("time","lat","lon") if d in da.dims]
    da = da.transpose(*order, ...)
    # month-end timestamps
    if "time" in da.dims:
        t = pd.to_datetime(da.time.values)
        t_end = pd.to_datetime([pd.Timestamp(tt).to_period("M").asfreq("M").to_timestamp() for tt in t])
        da = da.assign_coords(time=t_end)
    return da.sortby([c for c in ["lat","lon"] if c in da.dims])

def align_to_ref_grid(da: xr.DataArray, ref_lat: xr.DataArray, ref_lon: xr.DataArray) -> xr.DataArray:
    # nearest neighbor to preserve anomalies; sorts coords first
    da_sorted = da.sortby([c for c in ["lat","lon"] if c in da.dims])
    return da_sorted.interp(
        lat=ref_lat.sortby(ref_lat), lon=ref_lon.sortby(ref_lon),
        method="nearest", kwargs={"fill_value": np.nan}
    )

def corr_by_month(A: xr.DataArray, B: xr.DataArray) -> xr.DataArray:
    """
    Per calendar month (1..12), Pearson r across years at each grid cell:
      r_m(lat,lon) = corr( A[time in month m], B[time in month m] )
    Returns (month, lat, lon) with month=1..12.
    """
    if "time" not in A.dims or "time" not in B.dims:
        raise ValueError("Both arrays need a 'time' dimension.")
    months = np.arange(1, 13, dtype=int)
    outs = []
    for m in months:
        a_m = A.sel(time=A.time.dt.month == m)
        b_m = B.sel(time=B.time.dt.month == m)
        if a_m.sizes.get("time", 0) >= 3 and b_m.sizes.get("time", 0) >= 3:
            r = xr.corr(a_m, b_m, dim="time")
        else:
            r = A.isel(time=0, drop=True) * np.nan
        outs.append(r)
    R = xr.concat(outs, dim="month").assign_coords(month=("month", months))
    return R

def plot_corr_12panels(R: xr.DataArray, title: str, out_path: Path, gdf_boundary: gpd.GeoDataFrame):
    # Expect dims (month, lat, lon)
    R = R.transpose("month","lat","lon")
    lat = R["lat"].values; lon = R["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]

    cmap = get_cmap("rainbow").copy()
    cmap.set_bad("white")
    norm = Normalize(vmin=-1.0, vmax=1.0)

    fig, axes = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True, facecolor="white")
    axes = axes.ravel()
    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    for i in range(12):
        ax = axes[i]
        frame = R.sel(month=i+1)
        ax.imshow(frame.values, origin="lower", extent=extent, cmap=cmap, norm=norm, interpolation="nearest")
        gdf_boundary.boundary.plot(ax=ax, color="k", linewidth=0.6)
        ax.set_title(month_names[i], fontsize=11, color="black")
        ax.set_axis_off()
        ax.set_facecolor("white")

    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), orientation="vertical", fraction=0.03, pad=0.02)
    cbar.set_label("")  # no text label
    fig.suptitle(title, fontsize=14, color="black")
    fig.savefig(out_path, dpi=300, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)

# -------------------------
# India boundary
# -------------------------
gdf = gpd.read_file(INDIA_SHP)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS; set EPSG:4326.")
gdf = gdf.to_crs("EPSG:4326")

# -------------------------
# Load all drivers once (time normalized)
# -------------------------
drivers = {}
for p, vname in DRIVERS:
    drivers[vname] = open_var(p, vname)

# -------------------------
# Main loop: for each VDI product, correlate with each driver
# -------------------------
for vdi_path, vdi_title, out_tmpl in VDI_FILES:
    VDI = open_var(vdi_path, "VDI")  # final VDI
    if "time" not in VDI.dims:
        print(f"{vdi_path.name}: no 'time' dimension — skipping.")
        continue

    # Reference grid/time from this VDI
    ref_lat, ref_lon = VDI["lat"], VDI["lon"]
    t_vdi = set(VDI.time.values)

    for d_path, d_name in DRIVERS:
        Drv = drivers[d_name]

        # Intersect time
        if "time" not in Drv.dims:
            print(f"{d_name}: no time dimension — skipping.")
            continue
        common_time = np.array(sorted(t_vdi & set(Drv.time.values)))
        if len(common_time) < 36:  # <3 years of months
            print(f"Warning: {d_name} ∩ VDI overlap is small ({len(common_time)} months). Proceeding.")
        VDIc = VDI.sel(time=common_time)
        DRVc = Drv.sel(time=common_time)

        # Align driver to VDI grid
        DRVc = align_to_ref_grid(DRVc, ref_lat, ref_lon)

        # Compute per-month correlations across years
        R = corr_by_month(DRVc, VDIc)  # (month, lat, lon), r in [-1,1]

        # Plot & save
        out_png = VDI_DIR / out_tmpl.format(drv=d_name)
        plot_corr_12panels(R, f"Correlation: {d_name} vs {vdi_title}", out_png, gdf)


<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\71429325.py:2: SyntaxWarning: invalid escape sequence '\D'
  """
C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\71429325.py:115: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORR_ESI_dryz_vs_VDI_allbiomes.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\71429325.py:115: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORR_SMroot_dryz_vs_VDI_allbiomes.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\71429325.py:115: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


...

Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORR_SIF_dryz_vs_VDI_cropland.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\71429325.py:115: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORR_TVDI_Z_vs_VDI_cropland.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\71429325.py:115: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("rainbow").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\CORR_VOD_dryz_vs_VDI_cropland.png


In [47]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable, get_cmap

# -------------------------
# Paths
# -------------------------
VDI_DIR   = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
LULC_DIR  = Path(r"C:\Drought\Land Cover")
INDIA_SHP = LULC_DIR / "India_with_jk.shp"

# Final VDI products (already biome-specific)
VDI_FILES = [
    (VDI_DIR / "VDI_allbiomes.nc",    "VDI_allbiomes",   "AllIndia"),
    (VDI_DIR / "VDI_forest.nc",       "VDI_forest",      "Forest"),
    (VDI_DIR / "VDI_shrub_grass.nc",  "VDI_shrub_grass", "ShrubGrass"),
    (VDI_DIR / "VDI_cropland.nc",     "VDI_cropland",    "Cropland"),
]

# Drivers (to be masked by biomes for biome-specific maps)
DRIVERS = [
    (VDI_DIR / "ESI_dryz.nc",                                "ESI_dryz"),
    (VDI_DIR / "SMroot_dryz.nc",                             "SMroot_dryz"),
    (VDI_DIR / "SIF_dryz.nc",                                "SIF_dryz"),
    (VDI_DIR / "TVDI_Z_monthly_India_0p05deg_2001-2023.nc",  "TVDI_Z"),
    (VDI_DIR / "VOD_dryz.nc",                                "VOD_dryz"),
]

# Decades (inclusive)
DECADE1 = (2001, 2010)
DECADE2 = (2011, 2020)
MIN_DEN = 0.05  # minimum absolute baseline mean to compute % change; else NaN

# -------------------------
# Helpers: IO, alignment, masks
# -------------------------
def open_var(path: Path, preferred_var: str | None = None) -> xr.DataArray:
    ds = xr.open_dataset(path)
    if preferred_var and preferred_var in ds:
        da = ds[preferred_var]
    else:
        da = ds[list(ds.data_vars)[0]]
    # unify dims to (time, lat, lon)
    ren = {}
    for d in da.dims:
        dlow = d.lower()
        if dlow in ("y","latitude","lat"):  ren[d] = "lat"
        if dlow in ("x","longitude","lon"): ren[d] = "lon"
    if ren:
        da = da.rename(ren)
    order = [d for d in ("time","lat","lon") if d in da.dims]
    da = da.transpose(*order, ...)
    # normalize time to month-end
    if "time" in da.dims:
        t = pd.to_datetime(da.time.values)
        t_end = pd.to_datetime([pd.Timestamp(tt).to_period("M").asfreq("M").to_timestamp() for tt in t])
        da = da.assign_coords(time=t_end)
    return da.sortby([c for c in ["lat","lon"] if c in da.dims])

def align_to_ref_grid(da: xr.DataArray, ref_lat: xr.DataArray, ref_lon: xr.DataArray) -> xr.DataArray:
    # nearest (no smoothing)
    da_sorted = da.sortby([c for c in ["lat","lon"] if c in da.dims])
    return da_sorted.interp(
        lat=ref_lat.sortby(ref_lat), lon=ref_lon.sortby(ref_lon),
        method="nearest", kwargs={"fill_value": np.nan}
    )

def open_fraction(path: Path, ref_lat: xr.DataArray, ref_lon: xr.DataArray) -> xr.DataArray:
    ds = xr.open_dataset(path)
    var = "fraction" if "fraction" in ds else list(ds.data_vars)[0]
    da = ds[var]
    ren = {}
    for d in da.dims:
        dlow = d.lower()
        if dlow in ("y","latitude","lat"):  ren[d] = "lat"
        if dlow in ("x","longitude","lon"): ren[d] = "lon"
    if ren:
        da = da.rename(ren)
    da = da.transpose("lat","lon")
    return align_to_ref_grid(da, ref_lat, ref_lon).fillna(0.0).astype("float32")

# -------------------------
# India outline (for overlay)
# -------------------------
gdf = gpd.read_file(INDIA_SHP)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS; please set EPSG:4326.")
gdf = gdf.to_crs("EPSG:4326")

# -------------------------
# Load a reference grid (from VDI_allbiomes)
# -------------------------
ref_VDI = open_var(VDI_DIR / "VDI_allbiomes.nc", "VDI")
ref_lat, ref_lon = ref_VDI["lat"], ref_VDI["lon"]

# -------------------------
# Build biome masks (dominant class)
# -------------------------
frac_forest = open_fraction(LULC_DIR / "lulc_forest_frac_0p05deg.nc",      ref_lat, ref_lon)
frac_shrub  = open_fraction(LULC_DIR / "lulc_shrub_grass_frac_0p05deg.nc", ref_lat, ref_lon)
frac_crop   = open_fraction(LULC_DIR / "lulc_cropland_frac_0p05deg.nc",    ref_lat, ref_lon)

stacked = xr.concat([frac_forest, frac_shrub, frac_crop], dim="biome")  # 0,1,2
dom_idx = stacked.argmax("biome")
none = (stacked.sum("biome") <= 0)

biome_code = xr.full_like(frac_forest, np.nan, dtype="float32")
biome_code = xr.where(dom_idx==0, 1.0, biome_code)  # Forest
biome_code = xr.where(dom_idx==1, 2.0, biome_code)  # ShrubGrass
biome_code = xr.where(dom_idx==2, 3.0, biome_code)  # Cropland
biome_code = biome_code.where(~none)

MASKS = {
    "AllIndia": xr.ones_like(frac_forest, dtype="float32"),
    "Forest":   xr.where(biome_code==1.0, 1.0, 0.0),
    "ShrubGrass": xr.where(biome_code==2.0, 1.0, 0.0),
    "Cropland": xr.where(biome_code==3.0, 1.0, 0.0),
}

# -------------------------
# Decadal % change per month
# -------------------------
def percent_change_decadal(da: xr.DataArray, min_den: float = MIN_DEN) -> xr.DataArray:
    """
    Return 12-month % changes (2011–2020 vs 2001–2010) at each grid cell:
      %Δ_m = 100 * (mean_2011–2020(m) – mean_2001–2010(m)) / |mean_2001–2010(m)|
    dims: (month, lat, lon)
    NaN where baseline magnitude < min_den.
    """
    if "time" not in da.dims:
        raise ValueError("Input must have 'time' dimension (monthly).")

    months = np.arange(1, 13, dtype=int)
    out = []

    for m in months:
        # subset this month only
        msel = da.where(da.time.dt.month == m, drop=True)

        # years aligned to THIS subset (prevents boolean-length mismatch)
        y = msel.time.dt.year

        # split by decades using the subset’s own time index
        d1 = msel.where((y >= DECADE1[0]) & (y <= DECADE1[1]), drop=True)
        d2 = msel.where((y >= DECADE2[0]) & (y <= DECADE2[1]), drop=True)

        if d1.sizes.get("time", 0) == 0 or d2.sizes.get("time", 0) == 0:
            out.append(msel.isel(time=0, drop=True) * np.nan)
            continue

        mu1 = d1.mean("time", skipna=True)
        mu2 = d2.mean("time", skipna=True)

        den = np.abs(mu1)
        pct = xr.where(den >= min_den, 100.0 * (mu2 - mu1) / den, np.nan)
        out.append(pct)

    return xr.concat(out, dim="month").assign_coords(month=("month", months))
# -------------------------
# Plotting utility (12 panels)
# -------------------------
def plot_12panels(P: xr.DataArray, title: str, out_png: Path):
    # Expect dims (month, lat, lon)
    P = P.transpose("month","lat","lon")
    lat = P["lat"].values; lon = P["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]

    # Diverging palette: green (declining drought, negative) → red (rising drought, positive)
    cmap = get_cmap("RdYlGn_r").copy()
    cmap.set_bad("white")

    # Symmetric robust limits around 0
    lo, hi = np.nanpercentile(P.values, [2, 98])
    vmax = max(abs(lo), abs(hi))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 50.0  # fallback
    norm = Normalize(vmin=-vmax, vmax=+vmax)

    fig, axes = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True, facecolor="white")
    axes = axes.ravel()
    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    for i in range(12):
        ax = axes[i]
        frame = P.sel(month=i+1)
        ax.imshow(frame.values, origin="lower", extent=extent, cmap=cmap, norm=norm, interpolation="nearest")
        gdf.boundary.plot(ax=ax, color="k", linewidth=0.6)
        ax.set_title(month_names[i], fontsize=11, color="black")
        ax.set_axis_off()
        ax.set_facecolor("white")

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), orientation="vertical", fraction=0.03, pad=0.02)
    cbar.set_label("% change (2011–2020 vs 2001–2010)")  # short label

    fig.suptitle(title, fontsize=14, color="black")
    fig.savefig(out_png, dpi=300, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_png)

# -------------------------
# 1) VDI: each biome file already separate → compute & plot directly
# -------------------------
for vdi_path, vdi_tag, biome_tag in VDI_FILES:
    VDI = open_var(vdi_path, "VDI")
    # Ensure we only use the two decades
    years = pd.to_datetime(VDI.time.values).year
    VDI = VDI.sel(time=((years >= DECADE1[0]) & (years <= DECADE2[1])))

    P = percent_change_decadal(VDI)  # (month, lat, lon), %
    title = f"Decadal % change in VDI ({DECADE2[0]}–{DECADE2[1]} vs {DECADE1[0]}–{DECADE1[1]}) — {biome_tag}"
    out_png = VDI_DIR / f"VDI_{vdi_tag}_DecadalPctChange_{DECADE2[0]}-{DECADE2[1]}_vs_{DECADE1[0]}-{DECADE1[1]}.png"
    plot_12panels(P, title, out_png)

# -------------------------
# 2) Drivers: compute once, then plot for AllIndia and with biome masks
# -------------------------
# Load drivers and align to reference grid/time
drivers = {}
for p, vname in DRIVERS:
    da = open_var(p, vname)
    da = align_to_ref_grid(da, ref_lat, ref_lon)
    drivers[vname] = da

# Restrict to decades range and intersect common time across drivers (for consistency)
common_time = None
for v in drivers.values():
    yrs = pd.to_datetime(v.time.values).year
    vv = v.sel(time=((yrs >= DECADE1[0]) & (yrs <= DECADE2[1])))
    if common_time is None:
        common_time = set(vv.time.values)
    else:
        common_time &= set(vv.time.values)
for k in list(drivers.keys()):
    drivers[k] = drivers[k].sel(time=np.array(sorted(common_time)))

# Compute decadal % change per month for each driver
driver_pct = {k: percent_change_decadal(v) for k, v in drivers.items()}  # (month, lat, lon)

# Plot for AllIndia and biome-masked
for drv_name, P in driver_pct.items():
    for biome_tag, mask in MASKS.items():
        Pm = P.where(mask == 1.0)
        title = (f"Decadal % change in {drv_name} "
                 f"({DECADE2[0]}–{DECADE2[1]} vs {DECADE1[0]}–{DECADE1[1]}) — {biome_tag}")
        out_png = VDI_DIR / f"{drv_name}_DecadalPctChange_{biome_tag}_{DECADE2[0]}-{DECADE2[1]}_vs_{DECADE1[0]}-{DECADE1[1]}.png"
        plot_12panels(Pm, title, out_png)


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2549389455.py:195: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("RdYlGn_r").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_VDI_allbiomes_DecadalPctChange_2011-2020_vs_2001-2010.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2549389455.py:195: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("RdYlGn_r").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_VDI_forest_DecadalPctChange_2011-2020_vs_2001-2010.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2549389455.py:195: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("RdYlGn_r").copy()


...

Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VOD_dryz_DecadalPctChange_Forest_2011-2020_vs_2001-2010.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2549389455.py:195: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("RdYlGn_r").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VOD_dryz_DecadalPctChange_ShrubGrass_2011-2020_vs_2001-2010.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\2549389455.py:195: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("RdYlGn_r").copy()


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VOD_dryz_DecadalPctChange_Cropland_2011-2020_vs_2001-2010.png


In [49]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap, ScalarMappable

# -------------------------
# Paths
# -------------------------
VDI_DIR   = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
INDIA_SHP = Path(r"C:\Drought\Land Cover\India_with_jk.shp")

FILES = {
    "AllIndia":   VDI_DIR / "VDI_allbiomes.nc",
    "Forest":     VDI_DIR / "VDI_forest.nc",
    "ShrubGrass": VDI_DIR / "VDI_shrub_grass.nc",
    "Cropland":   VDI_DIR / "VDI_cropland.nc",
}

SEASONS = {
    "DJF":  [12, 1, 2],      # labeled by year of Jan/Feb
    "MAM":  [3, 4, 5],
    "JJAS": [6, 7, 8, 9],
    "ON":   [10, 11],
}

# -------------------------
# Helpers
# -------------------------
def open_vdi(path: Path) -> xr.DataArray:
    ds = xr.open_dataset(path)
    vname = "VDI" if "VDI" in ds else list(ds.data_vars)[0]
    da = ds[vname]
    # unify dims
    ren = {}
    for d in da.dims:
        dlow = d.lower()
        if dlow in ("y","latitude","lat"):  ren[d] = "lat"
        if dlow in ("x","longitude","lon"): ren[d] = "lon"
    if ren:
        da = da.rename(ren)
    # order
    da = da.transpose(*[d for d in ("time","lat","lon") if d in da.dims], ...)
    # standardize to month-end stamps
    if "time" in da.dims:
        t = pd.to_datetime(da.time.values)
        t_end = pd.to_datetime([pd.Timestamp(tt).to_period("M").asfreq("M").to_timestamp() for tt in t])
        da = da.assign_coords(time=t_end)
    return da.sortby(["lat","lon"])

def seasonal_means_interannual(da: xr.DataArray, season_months: list, season_name: str) -> xr.DataArray:
    """
    Build seasonal means per year: (year, lat, lon).
    For DJF: Dec is assigned to the following year (season year = year if month!=12 else year+1).
    """
    if "time" not in da.dims:
        raise ValueError("VDI needs a 'time' dimension.")
    # Subset to months in the season
    sel = da.where(da.time.dt.month.isin(season_months), drop=True)

    # Season-year labeling
    t = sel.time
    years = t.dt.year
    months = t.dt.month

    if season_name == "DJF":
        # Dec belongs to next year's DJF
        season_year = years + xr.where(months == 12, 1, 0)
    else:
        season_year = years

    # attach as coordinate
    sel = sel.assign_coords(season_year=("time", season_year.values))

    # mean across time within each season_year
    seas = sel.groupby("season_year").mean("time", skipna=True)  # (season_year, lat, lon)

    # sort by season_year
    seas = seas.sortby("season_year")
    return seas.rename({"season_year": "year"})

def robust_symmetric_limits(arr: np.ndarray, pct=(2,98), fallback=1.5):
    lo, hi = np.nanpercentile(arr, pct)
    vmax = max(abs(lo), abs(hi))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = fallback
    return -vmax, vmax

def make_interannual_figure(seas_da: xr.DataArray, season_name: str, biome_name: str, out_path: Path, gdf_boundary: gpd.GeoDataFrame):
    """
    seas_da: (year, lat, lon)
    """
    seas_da = seas_da.transpose("year","lat","lon")
    years = seas_da["year"].values
    n = len(years)
    if n == 0:
        print(f"[WARN] No data for {biome_name} {season_name}; skipping.")
        return

    # Layout
    ncols = min(5, n)
    nrows = math.ceil(n / ncols)

    # Extent
    lat = seas_da["lat"].values
    lon = seas_da["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]

    # Colormap & limits (symmetric robust around 0)
    cmap = get_cmap("RdYlGn_r").copy()  # green=wet, red=dry
    cmap.set_bad("white")
    vmin, vmax = robust_symmetric_limits(seas_da.values, pct=(2,98), fallback=2.5)
    norm = Normalize(vmin=vmin, vmax=vmax)

    # Figure
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2*ncols, 3.6*nrows),
                             constrained_layout=True, facecolor="white")
    axes = np.atleast_2d(axes).ravel()

    for i, yr in enumerate(years):
        ax = axes[i]
        frame = seas_da.sel(year=yr)
        ax.imshow(frame.values, origin="lower", extent=extent, cmap=cmap, norm=norm, interpolation="nearest")
        gdf_boundary.boundary.plot(ax=ax, color="k", linewidth=0.6)
        ax.set_title(f"{int(yr)}", fontsize=10, color="black")
        ax.set_axis_off()
        ax.set_facecolor("white")

    # hide any extra axes
    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    # Single colorbar on the right
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), orientation="vertical", fraction=0.025, pad=0.02)
    cbar.set_label("")  # no label text

    fig.suptitle(f"VDI Seasonal Interannual — {season_name} — {biome_name} (VDI>0 = drier)",
                 fontsize=14, color="black")
    fig.savefig(out_path, dpi=300, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)

# -------------------------
# India boundary
# -------------------------
gdf = gpd.read_file(INDIA_SHP)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS; please set EPSG:4326.")
gdf = gdf.to_crs("EPSG:4326")

# -------------------------
# Run for each biome and season
# -------------------------
for biome_name, nc_path in FILES.items():
    da = open_vdi(nc_path)   # (time, lat, lon)
    if "time" not in da.dims:
        print(f"[WARN] {nc_path.name} has no 'time'; skipping.")
        continue

    for season_name, months in SEASONS.items():
        seas = seasonal_means_interannual(da, months, season_name)   # (year, lat, lon)
        out_png = VDI_DIR / f"VDI_{biome_name}_{season_name}_interannual.png"
        make_interannual_figure(seas, season_name, biome_name, out_png, gdf)


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\537953197.py:137: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("RdYlGn_r").copy()  # green=wet, red=dry


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_AllIndia_DJF_interannual.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\537953197.py:137: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("RdYlGn_r").copy()  # green=wet, red=dry


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_AllIndia_MAM_interannual.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\537953197.py:137: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("RdYlGn_r").copy()  # green=wet, red=dry


...

Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_Cropland_MAM_interannual.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\537953197.py:137: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("RdYlGn_r").copy()  # green=wet, red=dry


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_Cropland_JJAS_interannual.png


C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\537953197.py:137: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("RdYlGn_r").copy()  # green=wet, red=dry


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_Cropland_ON_interannual.png


In [71]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd

# ----------------------------
# Paths (edit if different)
# ----------------------------
root = r"C:\Drought"

paths = {
    "VDI_dir": os.path.join(root, r"Regridding and data clipping\Final Analysis\VDI"),
    "PDSI":    os.path.join(root, r"Regridding and data clipping\PDSI", "TerraClimate_PDSI_0p05deg_INDIA.nc"),
    "VHI":     os.path.join(root, r"Regridding and data clipping\VHI", "MODIS_VCI_TCI_VHI_India_0p05deg_2001-2023.nc"),
    "SPI":     os.path.join(root, r"SPI", "SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc"),
    "GRACE":   os.path.join(root, r"Regridding and data clipping\GRACE", "GRACE_JPL_mascon_India_0p05deg_2003-2023.nc"),
    "LULC_dir":os.path.join(root, r"Land Cover"),
    "shp":     os.path.join(root, r"Land Cover", "India_with_jk.shp"),
    "out":     os.path.join(root, r"Maps_out")
}
os.makedirs(paths["out"], exist_ok=True)

# ----------------------------
# Utilities
# ----------------------------
def open_nc(fp):
    """Open NetCDF and standardize to lat/lon (ascending)."""
    ds = xr.open_dataset(fp)

    # Normalize coord names
    rename = {}
    if "latitude" in ds: rename["latitude"] = "lat"
    if "longitude" in ds: rename["longitude"] = "lon"
    if "y" in ds and "lat" not in ds: rename["y"] = "lat"
    if "x" in ds and "lon" not in ds: rename["x"] = "lon"
    if rename:
        ds = ds.rename(rename)

    # Ensure ascending coords
    if "lat" in ds.coords:
        ds = ds.sortby("lat")
    if "lon" in ds.coords:
        ds = ds.sortby("lon")
    return ds

def first_var(ds, candidates):
    """Pick the first variable present from candidates; else first 3D time/lat/lon var."""
    for v in candidates:
        if v in ds.data_vars:
            return v
        # case-insensitive
        low = [k.lower() for k in ds.data_vars]
        if v.lower() in low:
            return list(ds.data_vars)[low.index(v.lower())]
    for v in ds.data_vars:
        dims = ds[v].dims
        if "time" in dims and (("lat" in dims) or ("y" in dims)) and (("lon" in dims) or ("x" in dims)):
            return v
    raise ValueError(f"Could not find expected variable in {list(ds.data_vars)}")

# Seasons (custom ON=Oct-Nov)
SEASONS = {
    "DJF": [12, 1, 2],
    "MAM": [3, 4, 5],
    "JJAS":[6, 7, 8, 9],
    "ON":  [10, 11],
}

def seasonal_climatology(da):
    """Return seasonal climatology with dim 'season' in order DJF,MAM,JJAS,ON."""
    if "time" not in da.dims:
        raise ValueError("DataArray must have 'time' dimension for climatology.")
    if not np.issubdtype(da["time"].dtype, np.datetime64):
        da = da.assign_coords(time=pd.to_datetime(da["time"].values))
    outs = []
    for s, months in SEASONS.items():
        sel = da.sel(time=da["time"].dt.month.isin(months))
        outs.append(sel.mean("time", skipna=True).assign_coords(season=s))
    return xr.concat(outs, dim="season")

def apply_mask(da, mask, thr=0.5, interp_method="nearest"):
    """
    Apply biome mask (fraction >= thr) to DataArray da.
    If grid differs, interpolate mask to da grid.
    """
    # Sort
    if "lat" in da.coords: da = da.sortby("lat")
    if "lon" in da.coords: da = da.sortby("lon")
    if "lat" in mask.coords: mask = mask.sortby("lat")
    if "lon" in mask.coords: mask = mask.sortby("lon")
    # Collapse time in mask if present
    if "time" in mask.dims:
        mask = mask.mean("time")
    # Interp if sizes or coordinate values differ
    need_interp = (
        mask.sizes.get("lat", -1) != da.sizes.get("lat", -1) or
        mask.sizes.get("lon", -1) != da.sizes.get("lon", -1) or
        ("lat" in mask.coords and "lat" in da.coords and not np.array_equal(mask["lat"].values, da["lat"].values)) or
        ("lon" in mask.coords and "lon" in da.coords and not np.array_equal(mask["lon"].values, da["lon"].values))
    )
    if need_interp:
        mask = mask.interp(lat=da["lat"], lon=da["lon"], method=interp_method)
    mask_bool = xr.where(mask >= thr, 1.0, np.nan)
    return da * mask_bool

def plot_season_map_grid(da_season, title, out_png, shp_gdf, cmap="rainbow", vmin=None, vmax=None):
    """
    Plot 2x2 grid (DJF, MAM, JJAS, ON) with one right-side colorbar.
    Silences lat/lon ticks; white background; shapefile outline.
    """
    seasons = list(SEASONS.keys())
    proj = ccrs.PlateCarree()

    # color limits (robust)
    data_vals = da_season.values
    if vmin is None or vmax is None:
        vmin = float(np.nanpercentile(data_vals, 2))
        vmax = float(np.nanpercentile(data_vals, 98))
        if not np.isfinite(vmin) or not np.isfinite(vmax) or np.isclose(vmin, vmax):
            vmin = float(np.nanmin(data_vals))
            vmax = float(np.nanmax(data_vals))

    fig, axes = plt.subplots(2, 2, figsize=(10, 8), subplot_kw={"projection": proj})
    fig.patch.set_facecolor("white")
    axes = axes.ravel()

    mappable = None
    for i, s in enumerate(seasons):
        ax = axes[i]
        ax.set_facecolor("white")
        # draw
        im = ax.pcolormesh(da_season["lon"], da_season["lat"], da_season.sel(season=s),
                           transform=proj, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
        mappable = im
        # overlay boundary
        shp_gdf.boundary.plot(ax=ax, transform=proj, linewidth=0.6, edgecolor="black")
        # remove ticks/labels/frame
        ax.set_xticks([]); ax.set_yticks([])
        ax.spines["geo"].set_visible(False) if hasattr(ax, "spines") else None
        ax.set_title(s, fontsize=11, pad=2)

    # Single colorbar on the right
    cax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
    cb = fig.colorbar(mappable, cax=cax)
    cb.ax.tick_params(labelsize=9)

    fig.suptitle(title, fontsize=13)
    plt.subplots_adjust(left=0.03, right=0.88, top=0.92, bottom=0.05, wspace=0.02, hspace=0.08)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# ----------------------------
# Load shapefile (outline)
# ----------------------------
india_gdf = gpd.read_file(paths["shp"]).to_crs(epsg=4326)

# ----------------------------
# Load biome fraction masks
# ----------------------------
mask_files = {
    "cropland":    os.path.join(paths["LULC_dir"], "lulc_cropland_frac_0p05deg.nc"),
    "forest":      os.path.join(paths["LULC_dir"], "lulc_forest_frac_0p05deg.nc"),
    "shrub_grass": os.path.join(paths["LULC_dir"], "lulc_shrub_grass_frac_0p05deg.nc"),
}
biome_masks = {}
for name, fp in mask_files.items():
    ds = open_nc(fp)
    v = first_var(ds, ["fraction","frac","cropland","forest","shrub_grass","lulc"])
    m = ds[v]
    # standardize dims again (defensive)
    if "y" in m.dims and "lat" not in m.dims: m = m.rename({"y":"lat"})
    if "x" in m.dims and "lon" not in m.dims: m = m.rename({"x":"lon"})
    if "time" in m.dims: m = m.mean("time")
    biome_masks[name] = m

# ----------------------------
# VDI
# ----------------------------
print("Processing VDI...")
vdi_all = os.path.join(paths["VDI_dir"], "VDI_allbiomes.nc")
if not os.path.exists(vdi_all):
    # try a common alternative name from screenshot
    alt = [f for f in os.listdir(paths["VDI_dir"]) if f.lower().startswith("vdi_allbiomes") and f.lower().endswith(".nc")]
    if alt:
        vdi_all = os.path.join(paths["VDI_dir"], alt[0])
vdi_ds = open_nc(vdi_all)
vdi_var = first_var(vdi_ds, ["VDI","vdi"])
vdi = vdi_ds[vdi_var]
vdi_season = seasonal_climatology(vdi)
plot_season_map_grid(vdi_season, "VDI seasonal climatology – India",
                     os.path.join(paths["out"], "VDI_India_season.png"), india_gdf)

for bname, bmask in biome_masks.items():
    vdi_b = apply_mask(vdi, bmask, thr=0.5)
    plot_season_map_grid(seasonal_climatology(vdi_b),
                         f"VDI seasonal climatology – {bname.capitalize()}",
                         os.path.join(paths["out"], f"VDI_{bname}_season.png"), india_gdf)

# ----------------------------
# PDSI
# ----------------------------
print("Processing PDSI...")
pdsi_ds = open_nc(paths["PDSI"])
pdsi_var = first_var(pdsi_ds, ["PDSI","pdsi"])
pdsi = pdsi_ds[pdsi_var]
plot_season_map_grid(seasonal_climatology(pdsi), "PDSI seasonal climatology – India",
                     os.path.join(paths["out"], "PDSI_India_season.png"), india_gdf)
for bname, bmask in biome_masks.items():
    plot_season_map_grid(seasonal_climatology(apply_mask(pdsi, bmask)),
                         f"PDSI seasonal climatology – {bname.capitalize()}",
                         os.path.join(paths["out"], f"PDSI_{bname}_season.png"), india_gdf)

# ----------------------------
# VHI
# ----------------------------
print("Processing VHI...")
vhi_ds = open_nc(paths["VHI"])
vhi_var = first_var(vhi_ds, ["VHI","vhi"])
vhi = vhi_ds[vhi_var]
plot_season_map_grid(seasonal_climatology(vhi), "VHI seasonal climatology – India",
                     os.path.join(paths["out"], "VHI_India_season.png"), india_gdf)
for bname, bmask in biome_masks.items():
    plot_season_map_grid(seasonal_climatology(apply_mask(vhi, bmask)),
                         f"VHI seasonal climatology – {bname.capitalize()}",
                         os.path.join(paths["out"], f"VHI_{bname}_season.png"), india_gdf)

# ----------------------------
# SPI (use SPI3 and SPI6 only)
# ----------------------------
print("Processing SPI (3 & 6)...")
spi_ds = open_nc(paths["SPI"])
spi3_name = first_var(spi_ds, ["SPI3","SPI_3","spi3","spi_3"])
spi6_name = first_var(spi_ds, ["SPI6","SPI_6","spi6","spi_6"])

for var, label in [(spi3_name, "SPI3"), (spi6_name, "SPI6")]:
    spi = spi_ds[var]
    plot_season_map_grid(seasonal_climatology(spi), f"{label} seasonal climatology – India",
                         os.path.join(paths["out"], f"{label}_India_season.png"), india_gdf)
    for bname, bmask in biome_masks.items():
        plot_season_map_grid(seasonal_climatology(apply_mask(spi, bmask)),
                             f"{label} seasonal climatology – {bname.capitalize()}",
                             os.path.join(paths["out"], f"{label}_{bname}_season.png"), india_gdf)

# ----------------------------
# GRACE (e.g., TWS/TWSA)
# ----------------------------
print("Processing GRACE...")
grace_ds = open_nc(paths["GRACE"])
grace_var = first_var(grace_ds, ["TWS","GAB_msc_Lmax180","tws_anom","TWSA","grace","mascon"])
grace = grace_ds[grace_var]
plot_season_map_grid(seasonal_climatology(grace), "GRACE seasonal climatology – India",
                     os.path.join(paths["out"], "GRACE_India_season.png"), india_gdf)
for bname, bmask in biome_masks.items():
    plot_season_map_grid(seasonal_climatology(apply_mask(grace, bmask)),
                         f"GRACE seasonal climatology – {bname.capitalize()}",
                         os.path.join(paths["out"], f"GRACE_{bname}_season.png"), india_gdf)

print(f"Done. All figures saved to: {paths['out']}")


Processing VDI...
Processing PDSI...
Processing VHI...
Processing SPI (3 & 6)...
Processing GRACE...
Done. All figures saved to: C:\Drought\Maps_out


In [73]:
# ============================
# Monthly (12-panel) maps for selected drought years
# ============================

YEARS = [2002, 2005, 2009, 2015, 2019]
MAKE_INDIA = False  # set True if you also want India-wide monthly figures

MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

def monthly_means_for_year(da, year):
    """
    Create a 12-month DataArray for a given year:
    - each slice is the monthly mean for that year
    - months with no data become NaN
    returns DataArray with dims ('month','lat','lon') and coord 'month'=1..12
    """
    if "time" not in da.dims:
        raise ValueError("DataArray must have 'time' dimension.")
    if not np.issubdtype(da["time"].dtype, np.datetime64):
        da = da.assign_coords(time=pd.to_datetime(da["time"].values))

    frames = []
    # template for empty months
    template = da.isel(time=0, drop=True) * np.nan
    for m in range(1, 13):
        sel = da.sel(time=(da.time.dt.year == year) & (da.time.dt.month == m))
        if sel.sizes.get("time", 0) == 0:
            frames.append(template.assign_coords(month=m))
        else:
            frames.append(sel.mean("time", skipna=True).assign_coords(month=m))
    return xr.concat(frames, dim="month")

def plot_month_grid(da_month, title, out_png, shp_gdf, cmap="rainbow", vmin=None, vmax=None):
    """
    Plot a 3x4 grid for months Jan..Dec with one shared colorbar on the right.
    """
    proj = ccrs.PlateCarree()

    data_vals = da_month.values
    if vmin is None or vmax is None:
        vmin = float(np.nanpercentile(data_vals, 2))
        vmax = float(np.nanpercentile(data_vals, 98))
        if not np.isfinite(vmin) or not np.isfinite(vmax) or np.isclose(vmin, vmax):
            vmin = float(np.nanmin(data_vals))
            vmax = float(np.nanmax(data_vals))

    fig, axes = plt.subplots(3, 4, figsize=(12, 8), subplot_kw={"projection": proj})
    fig.patch.set_facecolor("white")
    axes = axes.ravel()

    mappable = None
    for i, m in enumerate(range(1, 13)):
        ax = axes[i]
        ax.set_facecolor("white")
        im = ax.pcolormesh(da_month["lon"], da_month["lat"], da_month.sel(month=m),
                           transform=proj, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
        mappable = im
        shp_gdf.boundary.plot(ax=ax, transform=proj, linewidth=0.6, edgecolor="black")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(MONTH_NAMES[m-1], fontsize=10, pad=2)

    # One colorbar
    cax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
    cb = fig.colorbar(mappable, cax=cax)
    cb.ax.tick_params(labelsize=9)

    fig.suptitle(title, fontsize=13)
    plt.subplots_adjust(left=0.03, right=0.88, top=0.92, bottom=0.05, wspace=0.02, hspace=0.08)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# Datasets already loaded above:
# vdi, pdsi, vhi, spi_ds[spi3_name], spi_ds[spi6_name], grace
datasets = [
    ("VDI",  vdi),
    ("PDSI", pdsi),
    ("VHI",  vhi),
    ("SPI3", spi_ds[spi3_name]),
    ("SPI6", spi_ds[spi6_name]),
    ("GRACE", grace),
]

print("Creating monthly 12-panel figures for drought years (biome-specific)...")
for name, da in datasets:
    for yr in YEARS:
        # (Optional) India-wide
        if MAKE_INDIA:
            m_da = monthly_means_for_year(da, yr)
            out_png = os.path.join(paths["out"], f"{name}_India_{yr}_monthly.png")
            plot_month_grid(m_da, f"{name} – {yr} monthly (India)", out_png, india_gdf)

        # Biome-specific maps
        for bname, bmask in biome_masks.items():
            m_da_biome = monthly_means_for_year(apply_mask(da, bmask, thr=0.5), yr)
            out_png_b = os.path.join(paths["out"], f"{name}_{bname}_{yr}_monthly.png")
            plot_month_grid(m_da_biome, f"{name} – {yr} monthly ({bname.capitalize()})", out_png_b, india_gdf)

print(f"Done monthly maps. Check: {paths['out']}")


Creating monthly 12-panel figures for drought years (biome-specific)...


C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\numpy\lib\nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1607956281.py:44: RuntimeWarning: All-NaN slice encountered
  vmin = float(np.nanmin(data_vals))
C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1607956281.py:45: RuntimeWarning: All-NaN slice encountered
  vmax = float(np.nanmax(data_vals))
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\numpy\lib\nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1607956281.py:44: RuntimeWarning: All-NaN slice encountered
  vmin = float(np.nanmin(data_vals))
C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_1436\1607956281.py:45: RuntimeWarning: All-NaN slice encountered
  vmax = float(np.nanmax(data_vals))
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\sit

Done monthly maps. Check: C:\Drought\Maps_out


In [3]:
#EDDI
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd

# ----------------------------
# Paths (edit root if needed)
# ----------------------------
root = r"C:\Drought"

paths = {
    "EDDI":     os.path.join(root, r"EDDI", "EDDI_005deg_200012-202312.nc"),
    "LULC_dir": os.path.join(root, r"Land Cover"),
    "shp":      os.path.join(root, r"Land Cover", "India_with_jk.shp"),
    "out":      os.path.join(root, r"Maps_out")
}
os.makedirs(paths["out"], exist_ok=True)

# ----------------------------
# Helpers
# ----------------------------
def open_nc(fp):
    """Open NetCDF and standardize to lat/lon (ascending)."""
    ds = xr.open_dataset(fp)

    # Normalize coord names
    rename = {}
    if "latitude" in ds: rename["latitude"] = "lat"
    if "longitude" in ds: rename["longitude"] = "lon"
    if "y" in ds and "lat" not in ds: rename["y"] = "lat"
    if "x" in ds and "lon" not in ds: rename["x"] = "lon"
    if rename:
        ds = ds.rename(rename)

    # Ensure ascending coords
    if "lat" in ds.coords: ds = ds.sortby("lat")
    if "lon" in ds.coords: ds = ds.sortby("lon")
    return ds

def first_var(ds, candidates):
    """Pick the first present variable from candidates (case-insensitive), else first 3D time/lat/lon var."""
    cand_lower = [c.lower() for c in candidates]
    for v in ds.data_vars:
        if v.lower() in cand_lower:
            return v
    for v in candidates:
        if v in ds.data_vars:
            return v
    # fallback: first 3D time/lat/lon var
    for v in ds.data_vars:
        dims = ds[v].dims
        if "time" in dims and (("lat" in dims) or ("y" in dims)) and (("lon" in dims) or ("x" in dims)):
            return v
    raise ValueError(f"Could not find expected variable in: {list(ds.data_vars)}")

SEASONS = {"DJF":[12,1,2], "MAM":[3,4,5], "JJAS":[6,7,8,9], "ON":[10,11]}

def seasonal_climatology(da):
    """Seasonal climatology with dim 'season' in order DJF, MAM, JJAS, ON."""
    if "time" not in da.dims:
        raise ValueError("DataArray must have 'time' dimension for climatology.")
    if not np.issubdtype(da["time"].dtype, np.datetime64):
        da = da.assign_coords(time=pd.to_datetime(da["time"].values))
    outs = []
    for s, months in SEASONS.items():
        sel = da.sel(time=da["time"].dt.month.isin(months))
        outs.append(sel.mean("time", skipna=True).assign_coords(season=s))
    return xr.concat(outs, dim="season")

def monthly_means_for_year(da, year):
    """
    12-month DataArray for a given year (monthly means).
    Forces float dtype; months with no data are NaN.
    Output dims: ('month','lat','lon'); month=1..12
    """
    if "time" not in da.dims:
        raise ValueError("DataArray must have 'time' dimension.")
    if not np.issubdtype(da["time"].dtype, np.datetime64):
        da = da.assign_coords(time=pd.to_datetime(da["time"].values))
    if not np.issubdtype(da.dtype, np.number):
        da = da.astype("float64")

    frames = []
    template = da.isel(time=0, drop=True)
    template = xr.full_like(template, np.nan, dtype="float64")

    for m in range(1, 13):
        sel = da.sel(time=(da.time.dt.year == year) & (da.time.dt.month == m))
        if sel.sizes.get("time", 0) == 0:
            frames.append(template.assign_coords(month=m))
        else:
            frames.append(sel.mean("time", skipna=True).astype("float64").assign_coords(month=m))
    out = xr.concat(frames, dim="month")
    if "month" not in out.coords:
        out = out.assign_coords(month=np.arange(1, 13))
    return out

def apply_mask(da, mask, thr=0.5, interp_method="nearest"):
    """
    Apply biome mask (fraction >= thr) to DataArray da.
    If grid differs, interpolate mask to da grid.
    """
    # Sort
    if "lat" in da.coords: da = da.sortby("lat")
    if "lon" in da.coords: da = da.sortby("lon")
    if "lat" in mask.coords: mask = mask.sortby("lat")
    if "lon" in mask.coords: mask = mask.sortby("lon")
    # Collapse time in mask if present
    if "time" in mask.dims:
        mask = mask.mean("time")
    # Interp if sizes or coordinate values differ
    need_interp = (
        mask.sizes.get("lat", -1) != da.sizes.get("lat", -1) or
        mask.sizes.get("lon", -1) != da.sizes.get("lon", -1) or
        ("lat" in mask.coords and "lat" in da.coords and not np.array_equal(mask["lat"].values, da["lat"].values)) or
        ("lon" in mask.coords and "lon" in da.coords and not np.array_equal(mask["lon"].values, da["lon"].values))
    )
    if need_interp:
        mask = mask.interp(lat=da["lat"], lon=da["lon"], method=interp_method)
    mask_bool = xr.where(mask >= thr, 1.0, np.nan)
    return da * mask_bool

def plot_season_map_grid(da_season, title, out_png, shp_gdf, cmap="rainbow", vmin=None, vmax=None):
    """
    Plot 2x2 grid (DJF, MAM, JJAS, ON) with one right-side colorbar.
    """
    seasons = list(SEASONS.keys())
    proj = ccrs.PlateCarree()

    # robust color limits
    arr = np.asarray(da_season.values, dtype=float)
    if vmin is None or vmax is None:
        finite = arr[np.isfinite(arr)]
        if finite.size >= 5:
            vmin = float(np.nanpercentile(finite, 2))
            vmax = float(np.nanpercentile(finite, 98))
            if not np.isfinite(vmin) or not np.isfinite(vmax) or np.isclose(vmin, vmax):
                vmin, vmax = float(np.nanmin(finite)), float(np.nanmax(finite))
        elif finite.size > 0:
            vmin, vmax = float(np.nanmin(finite)), float(np.nanmax(finite))
        else:
            vmin, vmax = -1.0, 1.0

    fig, axes = plt.subplots(2, 2, figsize=(10, 8), subplot_kw={"projection": proj})
    fig.patch.set_facecolor("white")
    axes = axes.ravel()

    mappable = None
    for i, s in enumerate(seasons):
        ax = axes[i]
        ax.set_facecolor("white")
        im = ax.pcolormesh(da_season["lon"], da_season["lat"], da_season.sel(season=s),
                           transform=proj, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
        mappable = im
        shp_gdf.boundary.plot(ax=ax, transform=proj, linewidth=0.6, edgecolor="black")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(s, fontsize=11, pad=2)

    # single colorbar
    cax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
    cb = fig.colorbar(mappable, cax=cax)
    cb.ax.tick_params(labelsize=9)

    fig.suptitle(title, fontsize=13)
    plt.subplots_adjust(left=0.03, right=0.88, top=0.92, bottom=0.05, wspace=0.02, hspace=0.08)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def plot_month_grid(da_month, title, out_png, shp_gdf, cmap="rainbow", vmin=None, vmax=None):
    """
    Plot a 3x4 grid for months Jan..Dec with one shared colorbar on the right.
    Robust vmin/vmax even if data contain NaNs or are entirely NaN.
    """
    proj = ccrs.PlateCarree()

    arr = np.asarray(da_month.to_numpy(), dtype=float)
    if vmin is None or vmax is None:
        finite = arr[np.isfinite(arr)]
        if finite.size >= 5:
            vmin = float(np.nanpercentile(finite, 2))
            vmax = float(np.nanpercentile(finite, 98))
            if not np.isfinite(vmin) or not np.isfinite(vmax) or np.isclose(vmin, vmax):
                vmin, vmax = float(np.nanmin(finite)), float(np.nanmax(finite))
        elif finite.size > 0:
            vmin, vmax = float(np.nanmin(finite)), float(np.nanmax(finite))
        else:
            vmin, vmax = -1.0, 1.0

    fig, axes = plt.subplots(3, 4, figsize=(12, 8), subplot_kw={"projection": proj})
    fig.patch.set_facecolor("white")
    axes = axes.ravel()

    months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    mappable = None
    for i, m in enumerate(range(1, 13)):
        ax = axes[i]
        ax.set_facecolor("white")
        im = ax.pcolormesh(da_month["lon"], da_month["lat"], da_month.sel(month=m),
                           transform=proj, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
        mappable = im
        shp_gdf.boundary.plot(ax=ax, transform=proj, linewidth=0.6, edgecolor="black")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(months[m-1], fontsize=10, pad=2)

    cax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
    cb = fig.colorbar(mappable, cax=cax)
    cb.ax.tick_params(labelsize=9)

    fig.suptitle(title, fontsize=13)
    plt.subplots_adjust(left=0.03, right=0.88, top=0.92, bottom=0.05, wspace=0.02, hspace=0.08)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# ----------------------------
# Load India boundary & biome masks
# ----------------------------
india_gdf = gpd.read_file(paths["shp"]).to_crs(epsg=4326)

mask_files = {
    "cropland":    os.path.join(paths["LULC_dir"], "lulc_cropland_frac_0p05deg.nc"),
    "forest":      os.path.join(paths["LULC_dir"], "lulc_forest_frac_0p05deg.nc"),
    "shrub_grass": os.path.join(paths["LULC_dir"], "lulc_shrub_grass_frac_0p05deg.nc"),
}
biome_masks = {}
for name, fp in mask_files.items():
    ds = open_nc(fp)
    v = first_var(ds, ["fraction","frac","cropland","forest","shrub_grass","lulc"])
    m = ds[v]
    if "y" in m.dims and "lat" not in m.dims: m = m.rename({"y":"lat"})
    if "x" in m.dims and "lon" not in m.dims: m = m.rename({"x":"lon"})
    if "time" in m.dims: m = m.mean("time")
    biome_masks[name] = m

# ----------------------------
# Open EDDI and pick EDDI3 + EDDI6 (force numeric)
# ----------------------------
print("Opening EDDI...")
eddi_ds = open_nc(paths["EDDI"])
eddi3_name = first_var(eddi_ds, ["EDDI_03","EDDI3","eddi_03","EDDI_3","eddi3"])
eddi6_name = first_var(eddi_ds, ["EDDI_06","EDDI6","eddi_06","EDDI_6","eddi6"])

eddi3 = eddi_ds[eddi3_name]
eddi6 = eddi_ds[eddi6_name]
if not np.issubdtype(eddi3.dtype, np.number): eddi3 = eddi3.astype("float64")
if not np.issubdtype(eddi6.dtype, np.number): eddi6 = eddi6.astype("float64")

# ----------------------------
# Seasonal climatology (India + biomes)
# ----------------------------
print("EDDI seasonal climatology (EDDI3 & EDDI6)...")
plot_season_map_grid(
    seasonal_climatology(eddi3),
    "EDDI3 seasonal climatology – India",
    os.path.join(paths["out"], "EDDI3_India_season.png"),
    india_gdf,
)
for bname, bmask in biome_masks.items():
    plot_season_map_grid(
        seasonal_climatology(apply_mask(eddi3, bmask, thr=0.5)),
        f"EDDI3 seasonal climatology – {bname.capitalize()}",
        os.path.join(paths["out"], f"EDDI3_{bname}_season.png"),
        india_gdf,
    )

plot_season_map_grid(
    seasonal_climatology(eddi6),
    "EDDI6 seasonal climatology – India",
    os.path.join(paths["out"], "EDDI6_India_season.png"),
    india_gdf,
)
for bname, bmask in biome_masks.items():
    plot_season_map_grid(
        seasonal_climatology(apply_mask(eddi6, bmask, thr=0.5)),
        f"EDDI6 seasonal climatology – {bname.capitalize()}",
        os.path.join(paths["out"], f"EDDI6_{bname}_season.png"),
        india_gdf,
    )

# ----------------------------
# Monthly 12-panel maps for drought years (biome-specific)
# ----------------------------
print("EDDI monthly maps for drought years...")
YEARS = [2002, 2005, 2009, 2015, 2019]
MAKE_INDIA = False  # set True if you also want India-wide 12-panels

for yr in YEARS:
    if MAKE_INDIA:
        m3 = monthly_means_for_year(eddi3, yr)
        plot_month_grid(m3, f"EDDI3 – {yr} monthly (India)",
                        os.path.join(paths["out"], f"EDDI3_India_{yr}_monthly.png"), india_gdf)
        m6 = monthly_means_for_year(eddi6, yr)
        plot_month_grid(m6, f"EDDI6 – {yr} monthly (India)",
                        os.path.join(paths["out"], f"EDDI6_India_{yr}_monthly.png"), india_gdf)

    for bname, bmask in biome_masks.items():
        m3b = monthly_means_for_year(apply_mask(eddi3, bmask, thr=0.5), yr)
        plot_month_grid(m3b, f"EDDI3 – {yr} monthly ({bname.capitalize()})",
                        os.path.join(paths["out"], f"EDDI3_{bname}_{yr}_monthly.png"), india_gdf)

        m6b = monthly_means_for_year(apply_mask(eddi6, bmask, thr=0.5), yr)
        plot_month_grid(m6b, f"EDDI6 – {yr} monthly ({bname.capitalize()})",
                        os.path.join(paths["out"], f"EDDI6_{bname}_{yr}_monthly.png"), india_gdf)

print(f"Done. All EDDI figures saved to: {paths['out']}")


C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


Opening EDDI...
EDDI seasonal climatology (EDDI3 & EDDI6)...
EDDI monthly maps for drought years...
Done. All EDDI figures saved to: C:\Drought\Maps_out


In [7]:
# -*- coding: utf-8 -*-
"""
Self-contained spatial correlation & lag-correlation analysis
Pairs: VDI vs {PDSI, VHI, SPI-3, SPI-6, EDDI-3, EDDI-6}
Monthly data, regridded to VDI pixels, per-pixel Pearson r.
Outputs:
  - Correlation_VDI_vs_others.png  (2x3 panels)
  - Correlation_VDI_vs_<VAR>_lags.png (1x3 panels for lags 1,2,3 months)
Style: white bg, rainbow cmap, one right colorbar, no lat/lon, India shapefile overlay.
"""

import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import cartopy.crs as ccrs
import warnings
warnings.filterwarnings("ignore")

# ----------------------------
# Paths (edit 'root' if needed)
# ----------------------------
root = r"C:\Drought"
paths = {
    "VDI_dir": os.path.join(root, r"Regridding and data clipping\Final Analysis\VDI"),
    "PDSI":    os.path.join(root, r"Regridding and data clipping\PDSI", "TerraClimate_PDSI_0p05deg_INDIA.nc"),
    "VHI":     os.path.join(root, r"Regridding and data clipping\VHI", "MODIS_VCI_TCI_VHI_India_0p05deg_2001-2023.nc"),
    "SPI":     os.path.join(root, r"SPI", "SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc"),
    "GRACE":   os.path.join(root, r"Regridding and data clipping\GRACE", "GRACE_JPL_mascon_India_0p05deg_2003-2023.nc"),
    "EDDI":    os.path.join(root, r"EDDI", "EDDI_005deg_200012-202312.nc"),
    "shp":     os.path.join(root, r"Land Cover", "India_with_jk.shp"),
    "out":     os.path.join(root, r"Maps_out"),
}
os.makedirs(paths["out"], exist_ok=True)

# ----------------------------
# Helpers
# ----------------------------
def open_nc(fp):
    """Open NetCDF, standardize to lat/lon coords (ascending)."""
    ds = xr.open_dataset(fp)
    rename = {}
    if "latitude" in ds: rename["latitude"] = "lat"
    if "longitude" in ds: rename["longitude"] = "lon"
    if "y" in ds and "lat" not in ds: rename["y"] = "lat"
    if "x" in ds and "lon" not in ds: rename["x"] = "lon"
    if rename: ds = ds.rename(rename)
    if "lat" in ds.coords: ds = ds.sortby("lat")
    if "lon" in ds.coords: ds = ds.sortby("lon")
    return ds

def first_var(ds, candidates):
    """Pick the first present var from candidates (case-insensitive) else first time/lat/lon var."""
    cl = [c.lower() for c in candidates]
    for v in ds.data_vars:
        if v.lower() in cl: return v
    for v in candidates:
        if v in ds.data_vars: return v
    for v in ds.data_vars:
        dims = ds[v].dims
        if "time" in dims and (("lat" in dims) or ("y" in dims)) and (("lon" in dims) or ("x" in dims)):
            return v
    raise ValueError(f"Could not find expected variable among: {list(ds.data_vars)}")

def ensure_datetime_monthly(da):
    """
    Ensure 'time' is datetime64 and **monthly**.
    Uses resample to month-start ('MS') means; robust across xarray/pandas versions.
    """
    if "time" not in da.dims:
        raise ValueError("Input DataArray must have 'time' dimension.")
    # Coerce to datetime64[ns] if possible
    if not np.issubdtype(da["time"].dtype, np.datetime64):
        da = da.assign_coords(time=pd.to_datetime(da["time"].values))
    da = da.sortby("time")

    # If non-numeric dtype, cast to float for safety
    if not np.issubdtype(da.dtype, np.number):
        da = da.astype("float64")

    # Monthly aggregation to month-start timestamps
    # This works whether data are daily, submonthly, or already monthly at arbitrary days.
    da_m = da.resample(time="MS").mean(skipna=True)

    # Drop leading/trailing all-NaN months (optional; keeps plots clean)
    if "time" in da_m.coords:
        # mask months where *all* pixels are NaN
        all_nan = ~np.isfinite(da_m).any(dim=[d for d in da_m.dims if d != "time"])
        if all_nan.any():
            kept = (~all_nan).values
            if kept.any():
                da_m = da_m.sel(time=da_m["time"].values[kept])

    return da_m

def interp_to_ref_grid(da, ref_lat, ref_lon, method="nearest"):
    """Interpolate da onto reference (lat,lon) grid."""
    if "lat" not in da.coords and "y" in da.coords: da = da.rename({"y":"lat"})
    if "lon" not in da.coords and "x" in da.coords: da = da.rename({"x":"lon"})
    if "lat" in da.coords: da = da.sortby("lat")
    if "lon" in da.coords: da = da.sortby("lon")
    same_lat = ("lat" in da.coords) and np.array_equal(da["lat"].values, ref_lat.values)
    same_lon = ("lon" in da.coords) and np.array_equal(da["lon"].values, ref_lon.values)
    if same_lat and same_lon:
        return da
    return da.interp(lat=ref_lat, lon=ref_lon, method=method)

def time_overlap(a, b):
    """Intersect time axis of two monthly DataArrays."""
    ta = pd.Index(pd.to_datetime(a["time"].values))
    tb = pd.Index(pd.to_datetime(b["time"].values))
    common = np.array(sorted(ta.intersection(tb)))
    return a.sel(time=common), b.sel(time=common)

def corr_over_time(a, b):
    """Per-pixel Pearson r over 'time'; assumes same grid & time."""
    a_dm = a - a.mean("time", skipna=True)
    b_dm = b - b.mean("time", skipna=True)
    return xr.corr(a_dm, b_dm, dim="time")

def plot_corr_grid(corr_list, titles, out_png, shp_gdf, cmap="rainbow"):
    """2x3 grid with one right-side colorbar; r in [-1,1]."""
    proj = ccrs.PlateCarree()
    vmin, vmax = -1.0, 1.0
    fig, axes = plt.subplots(2, 3, figsize=(13, 7), subplot_kw={"projection": proj})
    fig.patch.set_facecolor("white")
    axes = axes.ravel()
    mappable = None
    for ax, rmap, ttl in zip(axes, corr_list, titles):
        ax.set_facecolor("white")
        im = ax.pcolormesh(rmap["lon"], rmap["lat"], rmap, transform=proj,
                           cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
        mappable = im
        shp_gdf.boundary.plot(ax=ax, transform=proj, linewidth=0.6, edgecolor="black")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(ttl, fontsize=11, pad=2)
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.70])
    cb = fig.colorbar(mappable, cax=cax); cb.set_label("Pearson r", fontsize=10)
    cb.ax.tick_params(labelsize=9)
    fig.suptitle("Spatial correlation with VDI (monthly, overlapping period)", fontsize=14)
    plt.subplots_adjust(left=0.03, right=0.90, top=0.92, bottom=0.06, wspace=0.04, hspace=0.10)
    fig.savefig(out_png, dpi=300, bbox_inches="tight"); plt.close(fig)

def lagged_overlap(a, b, lag):
    """Shift b forward by 'lag' months: correlate a(t) with b(t-lag)."""
    b_shift = b.shift(time=lag)
    return time_overlap(a, b_shift)

def corr_with_vdi(other_rg, lag=0, min_months=24):
    """Per-pixel Pearson r of VDI with 'other_rg' at a given lag."""
    if lag == 0:
        a, b = time_overlap(vdi_m, other_rg)
    else:
        a, b = lagged_overlap(vdi_m, other_rg, lag)
    valid = xr.ufuncs.isfinite(a) & xr.ufuncs.isfinite(b)
    nobs = valid.sum("time")
    r = corr_over_time(a, b)
    return r.where(nobs >= min_months)

def plot_corr_grid_lags(corrs_by_lag, label, out_png, shp_gdf, cmap="rainbow"):
    """1x3 grid for lags 1,2,3 with one right colorbar; r in [-1,1]."""
    proj = ccrs.PlateCarree()
    vmin, vmax = -1.0, 1.0
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), subplot_kw={"projection": proj})
    fig.patch.set_facecolor("white")
    mappable = None
    for ax, lag in zip(axes, [1, 2, 3]):
        rmap = corrs_by_lag[lag]
        ax.set_facecolor("white")
        im = ax.pcolormesh(rmap["lon"], rmap["lat"], rmap, transform=proj,
                           cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
        mappable = im
        shp_gdf.boundary.plot(ax=ax, transform=proj, linewidth=0.6, edgecolor="black")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"Lag {lag} mo", fontsize=11, pad=2)
    cax = fig.add_axes([0.92, 0.20, 0.015, 0.60])
    cb = fig.colorbar(mappable, cax=cax); cb.set_label("Pearson r", fontsize=10)
    cb.ax.tick_params(labelsize=9)
    fig.suptitle(f"VDI vs {label} (lagged correlation)", fontsize=14)
    plt.subplots_adjust(left=0.03, right=0.90, top=0.88, bottom=0.08, wspace=0.05)
    fig.savefig(out_png, dpi=300, bbox_inches="tight"); plt.close(fig)

# ----------------------------
# Load data
# ----------------------------
print("Loading datasets...")
india_gdf = gpd.read_file(paths["shp"]).to_crs(epsg=4326)

# VDI (pick a plausible file within the directory)
candidates = [f for f in os.listdir(paths["VDI_dir"]) if f.lower().endswith(".nc")]
pref = [f for f in candidates if "vdi" in f.lower() and ("all" in f.lower() or "india" in f.lower())]
vdi_fp = os.path.join(paths["VDI_dir"], pref[0] if pref else candidates[0])

vdi_ds = open_nc(vdi_fp)
vdi_var = first_var(vdi_ds, ["VDI","vdi"])
vdi = vdi_ds[vdi_var]

pdsi_ds = open_nc(paths["PDSI"]); pdsi = pdsi_ds[first_var(pdsi_ds, ["PDSI","pdsi"])]
vhi_ds  = open_nc(paths["VHI"]);  vhi  = vhi_ds[first_var(vhi_ds,  ["VHI","vhi"])]

spi_ds  = open_nc(paths["SPI"])
spi3    = spi_ds[first_var(spi_ds, ["SPI3","SPI_3","spi3","spi_3"])]
spi6    = spi_ds[first_var(spi_ds, ["SPI6","SPI_6","spi6","spi_6"])]

grace_ds= open_nc(paths["GRACE"])
grace   = grace_ds[first_var(grace_ds, ["TWS","GAB_msc_Lmax180","tws_anom","TWSA","grace","mascon"])]

eddi_ds = open_nc(paths["EDDI"])
eddi3   = eddi_ds[first_var(eddi_ds, ["EDDI_03","EDDI3","eddi_03","EDDI_3","eddi3"])]
eddi6   = eddi_ds[first_var(eddi_ds, ["EDDI_06","EDDI6","eddi_06","EDDI_6","eddi6"])]

# Coerce to numeric dtype (robust)
for name, arr in [("VDI", vdi), ("PDSI", pdsi), ("VHI", vhi),
                  ("SPI3", spi3), ("SPI6", spi6), ("GRACE", grace),
                  ("EDDI3", eddi3), ("EDDI6", eddi6)]:
    if not np.issubdtype(arr.dtype, np.number):
        locals()[name.lower()] = arr.astype("float64")

# ----------------------------
# Prepare monthly and align to VDI grid
# ----------------------------
print("Preparing monthly series & grids...")
vdi_m   = ensure_datetime_monthly(vdi)
ref_lat, ref_lon = vdi_m["lat"], vdi_m["lon"]

pdsi_m  = interp_to_ref_grid(ensure_datetime_monthly(pdsi),  ref_lat, ref_lon)
vhi_m   = interp_to_ref_grid(ensure_datetime_monthly(vhi),   ref_lat, ref_lon)
spi3_m  = interp_to_ref_grid(ensure_datetime_monthly(spi3),  ref_lat, ref_lon)
spi6_m  = interp_to_ref_grid(ensure_datetime_monthly(spi6),  ref_lat, ref_lon)
grace_m = interp_to_ref_grid(ensure_datetime_monthly(grace), ref_lat, ref_lon)
eddi3_m = interp_to_ref_grid(ensure_datetime_monthly(eddi3), ref_lat, ref_lon)
eddi6_m = interp_to_ref_grid(ensure_datetime_monthly(eddi6), ref_lat, ref_lon)

# ----------------------------
# Spatial correlation (no lag)
# ----------------------------
print("Computing spatial correlations (no lag)...")
targets = [
    ("PDSI",  pdsi_m),
    ("VHI",   vhi_m),
    ("SPI-3", spi3_m),
    ("SPI-6", spi6_m),
    ("EDDI-3", eddi3_m),
    ("EDDI-6", eddi6_m),
]

corr_maps = []
titles = []
for label, da in targets:
    r = corr_with_vdi(da, lag=0, min_months=24)
    corr_maps.append(r)
    titles.append(f"VDI vs {label}")

out_corr_png = os.path.join(paths["out"], "Correlation_VDI_vs_others.png")
plot_corr_grid(corr_maps, titles, out_corr_png, india_gdf)
print(f"Saved: {out_corr_png}")

# ----------------------------
# Lagged correlations (1, 2, 3 months) – one figure per variable
# ----------------------------
print("Computing lagged correlations (1,2,3 months)...")
for label, da in targets:
    corrs = {lag: corr_with_vdi(da, lag=lag, min_months=24) for lag in [1, 2, 3]}
    out_png = os.path.join(paths["out"], f"Correlation_VDI_vs_{label.replace(' ','_')}_lags.png")
    plot_corr_grid_lags(corrs, label, out_png, india_gdf)
    print(f"Saved: {out_png}")

print("All correlation figures are ready.")


Loading datasets...
Preparing monthly series & grids...
Computing spatial correlations (no lag)...
Saved: C:\Drought\Maps_out\Correlation_VDI_vs_others.png
Computing lagged correlations (1,2,3 months)...
Saved: C:\Drought\Maps_out\Correlation_VDI_vs_PDSI_lags.png
Saved: C:\Drought\Maps_out\Correlation_VDI_vs_VHI_lags.png
Saved: C:\Drought\Maps_out\Correlation_VDI_vs_SPI-3_lags.png
Saved: C:\Drought\Maps_out\Correlation_VDI_vs_SPI-6_lags.png
Saved: C:\Drought\Maps_out\Correlation_VDI_vs_EDDI-3_lags.png
Saved: C:\Drought\Maps_out\Correlation_VDI_vs_EDDI-6_lags.png
All correlation figures are ready.


In [9]:
# -*- coding: utf-8 -*-
"""
Spatial correlation with sign-harmonization (no lag):
VDI vs {PDSI, VHI->(100-VHI or -VHI), SPI-3, SPI-6, EDDI-3, EDDI-6}
All variables transformed so **higher = drier** before correlation.
Output: C:\Drought\Maps_out\Correlation_VDI_vs_others_SIGNFIX.png
"""

import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import cartopy.crs as ccrs
import warnings
warnings.filterwarnings("ignore")

# ----------------------------
# Paths (edit 'root' if needed)
# ----------------------------
root = r"C:\Drought"
paths = {
    "VDI_dir": os.path.join(root, r"Regridding and data clipping\Final Analysis\VDI"),
    "PDSI":    os.path.join(root, r"Regridding and data clipping\PDSI", "TerraClimate_PDSI_0p05deg_INDIA.nc"),
    "VHI":     os.path.join(root, r"Regridding and data clipping\VHI", "MODIS_VCI_TCI_VHI_India_0p05deg_2001-2023.nc"),
    "SPI":     os.path.join(root, r"SPI", "SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc"),
    "GRACE":   os.path.join(root, r"Regridding and data clipping\GRACE", "GRACE_JPL_mascon_India_0p05deg_2003-2023.nc"),
    "EDDI":    os.path.join(root, r"EDDI", "EDDI_005deg_200012-202312.nc"),
    "shp":     os.path.join(root, r"Land Cover", "India_with_jk.shp"),
    "out":     os.path.join(root, r"Maps_out"),
}
os.makedirs(paths["out"], exist_ok=True)

# ----------------------------
# Helpers
# ----------------------------
def open_nc(fp):
    """Open NetCDF, standardize to lat/lon coords (ascending)."""
    ds = xr.open_dataset(fp)
    rename = {}
    if "latitude" in ds: rename["latitude"] = "lat"
    if "longitude" in ds: rename["longitude"] = "lon"
    if "y" in ds and "lat" not in ds: rename["y"] = "lat"
    if "x" in ds and "lon" not in ds: rename["x"] = "lon"
    if rename: ds = ds.rename(rename)
    if "lat" in ds.coords: ds = ds.sortby("lat")
    if "lon" in ds.coords: ds = ds.sortby("lon")
    return ds

def first_var(ds, candidates):
    """Pick the first present var from candidates (case-insensitive) else first time/lat/lon var."""
    cl = [c.lower() for c in candidates]
    for v in ds.data_vars:
        if v.lower() in cl: return v
    for v in candidates:
        if v in ds.data_vars: return v
    for v in ds.data_vars:
        dims = ds[v].dims
        if "time" in dims and (("lat" in dims) or ("y" in dims)) and (("lon" in dims) or ("x" in dims)):
            return v
    raise ValueError(f"Could not find expected variable among: {list(ds.data_vars)}")

def ensure_datetime_monthly(da):
    """
    Ensure 'time' is datetime64 and monthly.
    Uses resample to month-start ('MS') means; robust across xarray/pandas versions.
    """
    if "time" not in da.dims:
        raise ValueError("Input DataArray must have 'time' dimension.")
    if not np.issubdtype(da["time"].dtype, np.datetime64):
        da = da.assign_coords(time=pd.to_datetime(da["time"].values))
    da = da.sortby("time")
    if not np.issubdtype(da.dtype, np.number):
        da = da.astype("float64")
    da_m = da.resample(time="MS").mean(skipna=True)
    # drop months entirely NaN across space (tidy)
    if "time" in da_m.coords:
        all_nan = ~np.isfinite(da_m).any(dim=[d for d in da_m.dims if d != "time"])
        if all_nan.any():
            keep = (~all_nan).values
            if keep.any():
                da_m = da_m.sel(time=da_m["time"].values[keep])
    return da_m

def interp_to_ref_grid(da, ref_lat, ref_lon, method="nearest"):
    """Interpolate da onto reference (lat,lon) grid."""
    if "lat" not in da.coords and "y" in da.coords: da = da.rename({"y":"lat"})
    if "lon" not in da.coords and "x" in da.coords: da = da.rename({"x":"lon"})
    if "lat" in da.coords: da = da.sortby("lat")
    if "lon" in da.coords: da = da.sortby("lon")
    same_lat = ("lat" in da.coords) and np.array_equal(da["lat"].values, ref_lat.values)
    same_lon = ("lon" in da.coords) and np.array_equal(da["lon"].values, ref_lon.values)
    if same_lat and same_lon:
        return da
    return da.interp(lat=ref_lat, lon=ref_lon, method=method)

def time_overlap(a, b):
    """Intersect time axis of two monthly DataArrays."""
    ta = pd.Index(pd.to_datetime(a["time"].values))
    tb = pd.Index(pd.to_datetime(b["time"].values))
    common = np.array(sorted(ta.intersection(tb)))
    return a.sel(time=common), b.sel(time=common)

def corr_over_time(a, b):
    """Per-pixel Pearson r over 'time'; assumes same grid & time."""
    a_dm = a - a.mean("time", skipna=True)
    b_dm = b - b.mean("time", skipna=True)
    return xr.corr(a_dm, b_dm, dim="time")

def plot_corr_grid(corr_list, titles, out_png, shp_gdf, cmap="rainbow"):
    """2x3 grid with one right-side colorbar; r in [-1,1]."""
    proj = ccrs.PlateCarree()
    vmin, vmax = -1.0, 1.0
    fig, axes = plt.subplots(2, 3, figsize=(13, 7), subplot_kw={"projection": proj})
    fig.patch.set_facecolor("white")
    axes = axes.ravel()
    mappable = None
    for ax, rmap, ttl in zip(axes, corr_list, titles):
        ax.set_facecolor("white")
        im = ax.pcolormesh(rmap["lon"], rmap["lat"], rmap, transform=proj,
                           cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
        mappable = im
        shp_gdf.boundary.plot(ax=ax, transform=proj, linewidth=0.6, edgecolor="black")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(ttl, fontsize=11, pad=2)
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.70])
    cb = fig.colorbar(mappable, cax=cax); cb.set_label("Pearson r", fontsize=10)
    cb.ax.tick_params(labelsize=9)
    fig.suptitle("Spatial correlation with VDI (monthly, sign-harmonized, overlapping period)", fontsize=14)
    plt.subplots_adjust(left=0.03, right=0.90, top=0.92, bottom=0.06, wspace=0.04, hspace=0.10)
    fig.savefig(out_png, dpi=300, bbox_inches="tight"); plt.close(fig)

# ----------------------------
# Load data
# ----------------------------
print("Loading datasets...")
india_gdf = gpd.read_file(paths["shp"]).to_crs(epsg=4326)

# VDI (pick a plausible file)
cand = [f for f in os.listdir(paths["VDI_dir"]) if f.lower().endswith(".nc")]
pref = [f for f in cand if "vdi" in f.lower() and ("all" in f.lower() or "india" in f.lower())]
vdi_fp = os.path.join(paths["VDI_dir"], pref[0] if pref else cand[0])

vdi_ds = open_nc(vdi_fp); vdi = vdi_ds[first_var(vdi_ds, ["VDI","vdi"])]

pdsi_ds = open_nc(paths["PDSI"]); pdsi = pdsi_ds[first_var(pdsi_ds, ["PDSI","pdsi"])]
vhi_ds  = open_nc(paths["VHI"]);  vhi  = vhi_ds[first_var(vhi_ds,  ["VHI","vhi"])]
spi_ds  = open_nc(paths["SPI"])
spi3    = spi_ds[first_var(spi_ds, ["SPI3","SPI_3","spi3","spi_3"])]
spi6    = spi_ds[first_var(spi_ds, ["SPI6","SPI_6","spi6","spi_6"])]
grace_ds= open_nc(paths["GRACE"]); grace = grace_ds[first_var(grace_ds, ["TWS","tws","tws_anom","TWSA","GAB_msc_Lmax180","mascon"])]
eddi_ds = open_nc(paths["EDDI"])
eddi3   = eddi_ds[first_var(eddi_ds, ["EDDI_03","EDDI3","eddi_03","EDDI_3","eddi3"])]
eddi6   = eddi_ds[first_var(eddi_ds, ["EDDI_06","EDDI6","eddi_06","EDDI_6","eddi6"])]

# Ensure numeric
for key, arr in [("vdi", vdi), ("pdsi", pdsi), ("vhi", vhi),
                 ("spi3", spi3), ("spi6", spi6), ("grace", grace),
                 ("eddi3", eddi3), ("eddi6", eddi6)]:
    if not np.issubdtype(arr.dtype, np.number):
        locals()[key] = arr.astype("float64")

# ----------------------------
# Monthly & common grid (VDI grid)
# ----------------------------
print("Preparing monthly series & grids...")
vdi_m   = ensure_datetime_monthly(vdi)
ref_lat, ref_lon = vdi_m["lat"], vdi_m["lon"]

pdsi_m  = interp_to_ref_grid(ensure_datetime_monthly(pdsi),  ref_lat, ref_lon)
vhi_m   = interp_to_ref_grid(ensure_datetime_monthly(vhi),   ref_lat, ref_lon)
spi3_m  = interp_to_ref_grid(ensure_datetime_monthly(spi3),  ref_lat, ref_lon)
spi6_m  = interp_to_ref_grid(ensure_datetime_monthly(spi6),  ref_lat, ref_lon)
grace_m = interp_to_ref_grid(ensure_datetime_monthly(grace), ref_lat, ref_lon)
eddi3_m = interp_to_ref_grid(ensure_datetime_monthly(eddi3), ref_lat, ref_lon)
eddi6_m = interp_to_ref_grid(ensure_datetime_monthly(eddi6), ref_lat, ref_lon)

# ----------------------------
# Sign harmonization (all => higher = drier)
# ----------------------------
def to_drought_positive(da, kind):
    """
    Convert various indices so that larger values mean 'drier'.
    kind in {'vdi','pdsi','spi','vhi','grace','eddi'}
    """
    if kind == "vdi":
        return da  # already positive=dry
    if kind == "pdsi":
        return -da  # negative dry -> flip
    if kind == "spi":
        return -da  # negative dry -> flip
    if kind == "grace":
        return -da  # negative deficit -> flip
    if kind == "eddi":
        return da   # positive dry
    if kind == "vhi":
        # VHI is 0..100 typically; low = dry
        # If clearly in 0..100, use (100 - VHI); else fallback to -VHI
        vmin = float(da.min(skipna=True))
        vmax = float(da.max(skipna=True))
        if 0.0 <= vmin <= 100.0 and 0.0 <= vmax <= 100.0:
            return 100.0 - da
        else:
            return -da
    raise ValueError(f"Unknown kind: {kind}")

vdi_d    = to_drought_positive(vdi_m,   "vdi")
pdsi_d   = to_drought_positive(pdsi_m,  "pdsi")
vhi_d    = to_drought_positive(vhi_m,   "vhi")
spi3_d   = to_drought_positive(spi3_m,  "spi")
spi6_d   = to_drought_positive(spi6_m,  "spi")
grace_d  = to_drought_positive(grace_m, "grace")
eddi3_d  = to_drought_positive(eddi3_m, "eddi")
eddi6_d  = to_drought_positive(eddi6_m, "eddi")

# ----------------------------
# Spatial correlation (no lag), now with sign-harmonized variables
# ----------------------------
print("Computing sign-harmonized spatial correlations (no lag)...")

def corr_with_vdi(other):
    a, b = time_overlap(vdi_d, other)
    valid = xr.ufuncs.isfinite(a) & xr.ufuncs.isfinite(b)
    nobs = valid.sum("time")
    r = corr_over_time(a, b)
    return r.where(nobs >= 24)

targets = [
    ("PDSI (dry=+)",  pdsi_d),
    ("VHI drought",   vhi_d),
    ("SPI-3 (dry=+)", spi3_d),
    ("SPI-6 (dry=+)", spi6_d),
    ("EDDI-3",        eddi3_d),
    ("EDDI-6",        eddi6_d),
    # You can swap one panel to GRACE if desired:
    # ("GRACE (dry=+)", grace_d),
]

corr_maps = []
titles = []
for label, da in targets:
    r = corr_with_vdi(da)
    corr_maps.append(r)
    titles.append(f"VDI vs {label}")

out_corr_png = os.path.join(paths["out"], "Correlation_VDI_vs_others_SIGNFIX.png")
plot_corr_grid(corr_maps, titles, out_corr_png, india_gdf)
print(f"Saved: {out_corr_png}")

print("Done.")


Loading datasets...
Preparing monthly series & grids...
Computing sign-harmonized spatial correlations (no lag)...
Saved: C:\Drought\Maps_out\Correlation_VDI_vs_others_SIGNFIX.png
Done.


In [13]:
# -*- coding: utf-8 -*-
"""
Shifting-distribution figures for drought metrics (like the temperature schematic).
Four figures (India + 3 biomes), each with 4 panels (VDI, PDSI, SPI-3, EDDI-3).
Two periods: 2001–2010 vs 2011–2020. Choice of area-mean or pixel-wise samples.
"""

import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import ListedColormap
from scipy.stats import gaussian_kde

# ----------------------------
# Settings
# ----------------------------
root = r"C:\Drought"
paths = {
    "VDI_dir": os.path.join(root, r"Regridding and data clipping\Final Analysis\VDI"),
    "PDSI":    os.path.join(root, r"Regridding and data clipping\PDSI", "TerraClimate_PDSI_0p05deg_INDIA.nc"),
    "SPI":     os.path.join(root, r"SPI", "SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc"),
    "EDDI":    os.path.join(root, r"EDDI", "EDDI_005deg_200012-202312.nc"),
    "LULC_dir":os.path.join(root, r"Land Cover"),
    "out":     os.path.join(root, r"Maps_in"),
}
os.makedirs(paths["out"], exist_ok=True)

# choose area mean (True) or pixel-wise distributions (False)
USE_AREA_MEAN = False

PAST_YEARS    = (2001, 2010)
CURR_YEARS    = (2011, 2020)

# ----------------------------
# Helpers
# ----------------------------
def open_nc(fp):
    ds = xr.open_dataset(fp)
    rn = {}
    if "latitude" in ds: rn["latitude"] = "lat"
    if "longitude" in ds: rn["longitude"] = "lon"
    if "y" in ds and "lat" not in ds: rn["y"] = "lat"
    if "x" in ds and "lon" not in ds: rn["x"] = "lon"
    if rn: ds = ds.rename(rn)
    if "lat" in ds: ds = ds.sortby("lat")
    if "lon" in ds: ds = ds.sortby("lon")
    return ds

def first_var(ds, candidates):
    cl = [c.lower() for c in candidates]
    for v in ds.data_vars:
        if v.lower() in cl: return v
    for c in candidates:
        if c in ds.data_vars: return c
    for v in ds.data_vars:
        dims = ds[v].dims
        if "time" in dims and (("lat" in dims) or ("y" in dims)) and (("lon" in dims) or ("x" in dims)):
            return v
    raise ValueError(f"Expected var not found in {list(ds.data_vars)}")

def monthly(da):
    if "time" not in da.dims:
        raise ValueError("No 'time' dim.")
    if not np.issubdtype(da.time.dtype, np.datetime64):
        da = da.assign_coords(time=pd.to_datetime(da.time.values))
    da = da.sortby("time")
    if not np.issubdtype(da.dtype, np.number):
        da = da.astype("float64")
    out = da.resample(time="MS").mean(skipna=True)
    # drop months that are all-NaN across space
    mask = np.isfinite(out).any(dim=[d for d in out.dims if d != "time"])
    return out.sel(time=mask)

def interp_to(da, ref_lat, ref_lon):
    if "lat" not in da and "y" in da: da = da.rename({"y": "lat"})
    if "lon" not in da and "x" in da: da = da.rename({"x": "lon"})
    if "lat" in da: da = da.sortby("lat")
    if "lon" in da: da = da.sortby("lon")
    same = ("lat" in da and np.array_equal(da.lat.values, ref_lat.values) and
            "lon" in da and np.array_equal(da.lon.values, ref_lon.values))
    return da if same else da.interp(lat=ref_lat, lon=ref_lon, method="nearest")

def load_mask(fp):
    ds = open_nc(fp)
    v = first_var(ds, ["fraction","frac","cropland","forest","shrub_grass","lulc"])
    m = ds[v]
    if "time" in m.dims: m = m.mean("time")
    return m

def apply_mask(da, frac_mask, thr=0.5):
    m = frac_mask
    if "lat" in m: m = m.sortby("lat")
    if "lon" in m: m = m.sortby("lon")
    need_interp = (m.sizes.get("lat",-1)!=da.sizes.get("lat",-1) or
                   m.sizes.get("lon",-1)!=da.sizes.get("lon",-1) or
                   (not np.array_equal(m.get_index("lat"), da.get_index("lat"))) or
                   (not np.array_equal(m.get_index("lon"), da.get_index("lon"))))
    if need_interp:
        m = m.interp(lat=da["lat"], lon=da["lon"], method="nearest")
    m_bool = xr.where(m>=thr, 1.0, np.nan)
    return da*m_bool

def area_weighted_mean(da):
    lat = da["lat"]
    w = np.cos(np.deg2rad(lat))
    W, _ = xr.broadcast(w, da["lon"])
    stencil = np.isfinite(da).any("time")
    W = W.where(stencil, 0.0)
    return da.weighted(W).mean(("lat","lon"), skipna=True)

def to_dry_positive(da, kind):
    if kind=="vdi": return da
    if kind=="eddi": return da
    if kind=="pdsi": return -da
    if kind=="spi": return -da
    raise ValueError(kind)

def period_slice(da, years):
    y0,y1 = years
    return da.sel(time=(da.time.dt.year>=y0) & (da.time.dt.year<=y1))

def kde_curve(values, xgrid):
    # gaussian KDE with Scott’s rule, safe for small arrays
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 8:
        return np.zeros_like(xgrid)
    kde = gaussian_kde(values)
    y = kde.evaluate(xgrid)
    return y / np.trapz(y, xgrid)  # normalize to integrate to 1

def robust_range(vecs):
    data = np.concatenate([v[np.isfinite(v)] for v in vecs if v.size])
    if data.size < 10:
        return (-1.0, 1.0)
    lo, hi = np.nanpercentile(data, [2,98])
    # ensure symmetric so 0 stays centered
    L = max(abs(lo), abs(hi))
    return (-L, L)

# Color band helper
def draw_category_band(ax, thresholds, labels):
    """
    thresholds: sorted list (including -inf and +inf at ends) in dry-positive units
    labels: same length-1 list of names
    Colors: green->yellow->red scale
    """
    # custom gradient palette (wet to dry)
    cols = ListedColormap(["#1a9850","#66bd63","#a6d96a","#fdae61","#f46d43","#d73027"]).colors
    colors = []
    # map len(labels) to available colors (2..6). We’ll sample evenly.
    for i in np.linspace(0, len(cols)-1, num=len(labels)):
        colors.append(cols[int(round(i))])

    ymin, ymax = ax.get_ylim()
    for i in range(len(labels)):
        x0 = thresholds[i]; x1 = thresholds[i+1]
        ax.add_patch(Rectangle((x0, ymin), x1-x0, ymax-ymin, color=colors[i], alpha=0.15, lw=0))
        ax.text((x0+x1)/2, ymin + 0.05*(ymax-ymin), labels[i],
                ha="center", va="bottom", fontsize=8, color="black", alpha=0.8, rotation=0)
    ax.set_ylim(ymin, ymax)

# ----------------------------
# Load data & masks
# ----------------------------
print("Loading data...")
# VDI (pick a likely file)
vdi_files = [f for f in os.listdir(paths["VDI_dir"]) if f.lower().endswith(".nc")]
vdi_pref  = [f for f in vdi_files if "vdi" in f.lower() and ("all" in f.lower() or "india" in f.lower())]
vdi_fp    = os.path.join(paths["VDI_dir"], vdi_pref[0] if vdi_pref else vdi_files[0])
vdi_ds    = open_nc(vdi_fp); vdi = vdi_ds[first_var(vdi_ds, ["VDI","vdi"])]

pdsi_ds   = open_nc(paths["PDSI"]); pdsi = pdsi_ds[first_var(pdsi_ds, ["PDSI","pdsi"])]
spi_ds    = open_nc(paths["SPI"]);  spi3  = spi_ds[first_var(spi_ds, ["SPI3","SPI_3","spi3","spi_3"])]
eddi_ds   = open_nc(paths["EDDI"]); eddi3 = eddi_ds[first_var(eddi_ds, ["EDDI_03","EDDI3","eddi_03","EDDI_3","eddi3"])]

# Monthly & common grid
vdi_m   = monthly(vdi)
ref_lat, ref_lon = vdi_m["lat"], vdi_m["lon"]
pdsi_m  = interp_to(monthly(pdsi), ref_lat, ref_lon)
spi3_m  = interp_to(monthly(spi3), ref_lat, ref_lon)
eddi3_m = interp_to(monthly(eddi3), ref_lat, ref_lon)

# Convert to “dry positive”
vdi_d   = to_dry_positive(vdi_m,   "vdi")
pdsi_d  = to_dry_positive(pdsi_m,  "pdsi")
spi3_d  = to_dry_positive(spi3_m,  "spi")
eddi3_d = to_dry_positive(eddi3_m, "eddi")

# Biome masks
masks = {
    "India":       None,
    "cropland":    load_mask(os.path.join(paths["LULC_dir"], "lulc_cropland_frac_0p05deg.nc")),
    "forest":      load_mask(os.path.join(paths["LULC_dir"], "lulc_forest_frac_0p05deg.nc")),
    "shrub_grass": load_mask(os.path.join(paths["LULC_dir"], "lulc_shrub_grass_frac_0p05deg.nc")),
}

# ----------------------------
# Category thresholds (dry-positive units)
# ----------------------------
# SPI/EDDI use standard sigma-like thresholds after sign flip:
SPI_EDDI_THRESH = [ -np.inf, 0.0, 0.5, 1.0, 1.5, 2.0,  np.inf]
SPI_EDDI_LABELS = ["Wet", "Near-norm", "Mild", "Moderate", "Severe", "Extreme"]

# PDSI (after sign flip): 0..1 near-normal, 1..2 incipient, 2..3 moderate, 3..4 severe, >=4 extreme
PDSI_THRESH     = [ -np.inf, 0.0, 1.0, 2.0, 3.0, 4.0,  np.inf]
PDSI_LABELS     = ["Wet", "Near-norm", "Incipient", "Moderate", "Severe", "Extreme"]

# VDI doesn’t have a universal scale; we’ll use sigma-like bins on the *within-figure* x-range
# We will compute panel-specific thresholds centered at 0 later.

# ----------------------------
# Plot per region
# ----------------------------
def collect_samples(da_drypos, mask_name, mask):
    """Return two 1D arrays: values for 2001–2010 and 2011–2020 (area mean or pixels)."""
    if mask_name != "India":
        da = apply_mask(da_drypos, mask, thr=0.5)
    else:
        da = da_drypos

    if USE_AREA_MEAN:
        ts = area_weighted_mean(da)  # 1D monthly
        past = period_slice(ts, PAST_YEARS).values
        curr = period_slice(ts, CURR_YEARS).values
    else:
        past = period_slice(da, PAST_YEARS).values.reshape(-1)
        curr = period_slice(da, CURR_YEARS).values.reshape(-1)

    return past[np.isfinite(past)], curr[np.isfinite(curr)]

def panel(ax, name, past_vals, curr_vals, fixed_thresholds=None, fixed_labels=None,
          color_past="#547CBA", color_curr="#D7772A"):
    """Draw KDE curves + category band."""
    # x-range
    xmin, xmax = robust_range([past_vals, curr_vals])
    x = np.linspace(xmin, xmax, 600)

    y1 = kde_curve(past_vals, x)
    y2 = kde_curve(curr_vals, x)

    # fill curves (past solid, current dashed edge)
    ax.fill_between(x, y1, color=color_past, alpha=0.35, label=f"{PAST_YEARS[0]}–{PAST_YEARS[1]}")
    ax.plot(x, y1, color=color_past, lw=1.8)

    ax.fill_between(x, y2, color=color_curr, alpha=0.25, label=f"{CURR_YEARS[0]}–{CURR_YEARS[1]}")
    ax.plot(x, y2, color=color_curr, lw=2.0, ls="--")

    # Category band (green→yellow→red)
    if fixed_thresholds is not None:
        thresholds = fixed_thresholds
        labels = fixed_labels
    else:
        # Dynamic “sigma-like” thresholds for VDI using symmetrical bins
        # [-inf, -2, -1.5, -1, -0.5, 0, 0.5, 1, 1.5, 2, inf] but we merge symmetric wet bins
        # Final: Wet (<0), Near-norm (0–0.5), Mild (0.5–1), Moderate (1–1.5), Severe (1.5–2), Extreme (>2)
        thresholds = [ -np.inf, 0.0, 0.5, 1.0, 1.5, 2.0,  np.inf]
        labels     = ["Wet", "Near-norm", "Mild", "Moderate", "Severe", "Extreme"]

    # draw band *after* we know y-limits
    ymin, ymax = ax.get_ylim()
    draw_category_band(ax, thresholds, labels)

    # cosmetics
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Drought intensity (dry +)")
    ax.set_yticks([])
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

def make_figure(region_name, mask):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.patch.set_facecolor("white")
    axes = axes.ravel()

    # Collect
    vdi_p,  vdi_c  = collect_samples(vdi_d,  region_name, mask)
    pdsi_p, pdsi_c = collect_samples(pdsi_d, region_name, mask)
    spi_p,  spi_c  = collect_samples(spi3_d, region_name, mask)
    eddi_p, eddi_c = collect_samples(eddi3_d, region_name, mask)

    # Panels
    panel(axes[0], "VDI",  vdi_p,  vdi_c, fixed_thresholds=None, fixed_labels=None)
    panel(axes[1], "PDSI (dry=+)", pdsi_p, pdsi_c, fixed_thresholds=PDSI_THRESH, fixed_labels=PDSI_LABELS)
    panel(axes[2], "SPI-3 (dry=+)",  spi_p,  spi_c, fixed_thresholds=SPI_EDDI_THRESH, fixed_labels=SPI_EDDI_LABELS)
    panel(axes[3], "EDDI-3",         eddi_p, eddi_c, fixed_thresholds=SPI_EDDI_THRESH, fixed_labels=SPI_EDDI_LABELS)

    # Single legend (top center)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, fontsize=10, bbox_to_anchor=(0.5, 1.02))

    fig.suptitle(f"{region_name} — Distribution shift of drought metrics (2001–2010 vs 2011–2020)", fontsize=14)
    plt.subplots_adjust(top=0.88, wspace=0.22, hspace=0.28, left=0.06, right=0.97, bottom=0.08)

    tag = "areamean" if USE_AREA_MEAN else "pixels"
    outpng = os.path.join(paths["out"], f"Drought_Distributions_{region_name}_{tag}.png")
    fig.savefig(outpng, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {outpng}")

# ----------------------------
# Build figures: India + 3 biomes
# ----------------------------
print("Creating figures...")
for region, m in masks.items():
    make_figure("India" if region=="India" else region.capitalize(), m)

print("Done.")


Loading data...
Creating figures...
Saved: C:\Drought\Maps_in\Drought_Distributions_India_pixels.png
Saved: C:\Drought\Maps_in\Drought_Distributions_Cropland_pixels.png
Saved: C:\Drought\Maps_in\Drought_Distributions_Forest_pixels.png
Saved: C:\Drought\Maps_in\Drought_Distributions_Shrub_grass_pixels.png
Done.
